In [ ]:
# === Cell 1: Data Root ===
import os

# Data root; override with the KI_SORT_ROOT environment variable.
KI_SORT_ROOT = os.environ.get("KI_SORT_ROOT", "/VOL_D/fio.h")


In [ ]:
# === Cell 2: Prerequisites ===
# Conda environment yolo-train with packages, registered as a Jupyter kernel.
"""
conda create -n yolo-train python=3.10 -y
conda activate yolo-train

conda install -y -c conda-forge \
  ipykernel numpy pandas pyyaml tqdm matplotlib scikit-learn psutil

pip install --no-cache-dir --index-url https://download.pytorch.org/whl/cu121 \
  torch torchvision torchaudio

pip install --no-cache-dir \
  ultralytics opencv-python-headless seaborn tensorboard ipywidgets \
  "faster-coco-eval>=1.6.7" statsmodels scipy

python -m ipykernel install --user --name yolo-train --display-name "yolo-train"
"""

In [ ]:
# === Cell 3: Imports ===

%matplotlib inline

import os
import sys
import json
import math
import random
import platform
import re
import time
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import yaml

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

import cv2
import torch
import ultralytics
from ultralytics import YOLO

import ipywidgets as widgets
from IPython.display import display, clear_output

from tqdm.auto import tqdm

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"]       = 120
plt.rcParams["savefig.dpi"]      = 150
plt.rcParams["figure.figsize"]   = (8, 5)
plt.rcParams["axes.titlesize"]   = 12
plt.rcParams["axes.labelsize"]   = 11
plt.rcParams["legend.fontsize"]  = 10
plt.rcParams["xtick.labelsize"]  = 10
plt.rcParams["ytick.labelsize"]  = 10

In [ ]:
# === Cell 4: Config ===

# --- Project paths ---
EXPORT_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/01_LS_Export/20260518_Export")
TRAIN_ROOT  = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/02_Image_Segmentation/20260527_Yolo_Model_Screening")

IMAGES_DIR = EXPORT_ROOT / "images" / "all"
LABELS_DIR = EXPORT_ROOT / "labels" / "all"

OBJECT_SPLIT_CSV = EXPORT_ROOT / "object_split.csv"
METADATA_AGG_CSV = EXPORT_ROOT / "metadata_aggregated.csv"
BASE_DATA_YAML   = EXPORT_ROOT / "data.yaml"

# --- Output folders ---
TRAIN_ROOT.mkdir(parents=True, exist_ok=True)

SPLITS_DIR    = TRAIN_ROOT / "splits"
RUNS_DIR      = TRAIN_ROOT / "runs"
RESULTS_DIR   = TRAIN_ROOT / "results"
INFERENCE_DIR = TRAIN_ROOT / "inference"

for p in [SPLITS_DIR, RUNS_DIR, RESULTS_DIR, INFERENCE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

PROJECT_NAME    = TRAIN_ROOT.name
EXPERIMENT_NAME = "20260527_Yolo_Model_Screening"

# --- Dataset semantics ---
CLASS_MAP = {
    "body":  0,
    "cap":   1,
    "label": 2,
}
MATERIAL_CLASSES = ["HDPE", "PET", "PP", "PS"]

# --- Reproducibility and execution ---
RANDOM_SEED = 42
DEVICE      = 0           # single GPU
WORKERS     = 4           # suggested maximum on the DGX server

# --- Cross-validation configuration ---
NUM_FOLDS    = 4
FOLDS_TO_RUN = "all"      # "all" or list of fold indices

# --- YOLO model screening list ---
YOLO_MODELS = [
    # YOLOv8 segmentation
    "yolov8n-seg.pt",
    "yolov8s-seg.pt",
    "yolov8m-seg.pt",
    #"yolov8l-seg.pt",
    #"yolov8x-seg.pt",

    # YOLO11 segmentation
    "yolo11n-seg.pt",
    "yolo11s-seg.pt",
    "yolo11m-seg.pt",
    #"yolo11l-seg.pt",
    #"yolo11x-seg.pt",

    # YOLO26 segmentation
    "yolo26n-seg.pt",
    "yolo26s-seg.pt",
    "yolo26m-seg.pt",
    #"yolo26l-seg.pt",
    #"yolo26x-seg.pt",
]

TASK = "segment"

# --- Training setup ---
TRAIN_EPOCHS   = 200
TRAIN_PATIENCE = 50

# Fixed batch size: with ~240-320 training images AutoBatch selects ~136, i.e. only
# 2 gradient steps per epoch, which prevents convergence.
BATCH_SIZE = 16
IMAGE_SIZE = 640
IMAGE_RECT = True

# Base learning rate (used with optimizer="SGD"; see Training cell)
TRAIN_LR0 = 0.01

# --- Augmentation profiles (DOE factor, not changed in screening) ---
AUGMENTATION_PROFILES = {
    "none": {
        "augment": False,
    },
    "standard": {
        "augment": True,
        # Add more keys here if needed, e.g.:
        # "hsv_h": 0.015,
        # "hsv_s": 0.7,
        # "hsv_v": 0.4,
        # "flipud": 0.5,
        # "fliplr": 0.5,
        # "scale": 0.5,
    },
}
AUG_PROFILE_NAME = "none"

# --- Metrics for best-epoch and model selection ---
BEST_EPOCH_METRIC           = "metrics/mAP50-95(M)"
BEST_MODEL_SELECTION_METRIC = "metrics/mAP50-95(M)"
F1_METRIC_NAME              = "metrics/F1(M)"

# --- Inference / evaluation thresholds ---
CONF_THRESHOLD = 0.25
IOU_THRESHOLD  = 0.50
MAX_DET        = 300

# --- Deterministic seeds ---
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# --- Print config ---
print("Configuration")
print("-" * 80)
print("EXPORT_ROOT:                ", EXPORT_ROOT)
print("TRAIN_ROOT:                 ", TRAIN_ROOT)
print("IMAGES_DIR:                 ", IMAGES_DIR)
print("LABELS_DIR:                 ", LABELS_DIR)
print("OBJECT_SPLIT_CSV:           ", OBJECT_SPLIT_CSV)
print("METADATA_AGG_CSV:           ", METADATA_AGG_CSV)
print("BASE_DATA_YAML:             ", BASE_DATA_YAML)
print("SPLITS_DIR:                 ", SPLITS_DIR)
print("RUNS_DIR:                   ", RUNS_DIR)
print("RESULTS_DIR:                ", RESULTS_DIR)
print("INFERENCE_DIR:              ", INFERENCE_DIR)
print("PROJECT_NAME:               ", PROJECT_NAME)
print("EXPERIMENT_NAME:            ", EXPERIMENT_NAME)
print("CLASS_MAP:                  ", CLASS_MAP)
print("MATERIAL_CLASSES:           ", MATERIAL_CLASSES)
print("RANDOM_SEED:                ", RANDOM_SEED)
print("DEVICE:                     ", DEVICE)
print("WORKERS:                    ", WORKERS)
print("NUM_FOLDS:                  ", NUM_FOLDS)
print("FOLDS_TO_RUN:               ", FOLDS_TO_RUN)
print("YOLO_MODELS:                ", YOLO_MODELS)
print("TASK:                       ", TASK)
print("TRAIN_EPOCHS:               ", TRAIN_EPOCHS)
print("TRAIN_PATIENCE:             ", TRAIN_PATIENCE)
print("BATCH_SIZE:                 ", BATCH_SIZE)
print("IMAGE_SIZE:                 ", IMAGE_SIZE)
print("IMAGE_RECT:                 ", IMAGE_RECT)
print("TRAIN_LR0:                  ", TRAIN_LR0)
print("AUG_PROFILE_NAME:           ", AUG_PROFILE_NAME)
print("BEST_EPOCH_METRIC:          ", BEST_EPOCH_METRIC)
print("BEST_MODEL_SELECTION_METRIC:", BEST_MODEL_SELECTION_METRIC)
print("F1_METRIC_NAME:             ", F1_METRIC_NAME)
print("CONF_THRESHOLD:             ", CONF_THRESHOLD)
print("IOU_THRESHOLD:              ", IOU_THRESHOLD)
print("MAX_DET:                    ", MAX_DET)

In [ ]:
# === Cell 5: Sanity Check ===

print("Python:            ", sys.version)
print("Platform:          ", platform.platform())
print("torch version:     ", torch.__version__)
print("CUDA available:    ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device count: ", torch.cuda.device_count())
    print("CUDA device 0:     ", torch.cuda.get_device_name(0))
print("ultralytics version:", ultralytics.__version__)

PATHS_TO_CHECK = [
    ("EXPORT_ROOT",      EXPORT_ROOT),
    ("TRAIN_ROOT",       TRAIN_ROOT),
    ("IMAGES_DIR",       IMAGES_DIR),
    ("LABELS_DIR",       LABELS_DIR),
    ("OBJECT_SPLIT_CSV", OBJECT_SPLIT_CSV),
    ("METADATA_AGG_CSV", METADATA_AGG_CSV),
    ("BASE_DATA_YAML",   BASE_DATA_YAML),
    ("SPLITS_DIR",       SPLITS_DIR),
    ("RUNS_DIR",         RUNS_DIR),
    ("RESULTS_DIR",      RESULTS_DIR),
    ("INFERENCE_DIR",    INFERENCE_DIR),
]

all_paths_ok = True
print("\nPath checks")
for path_name, path_value in PATHS_TO_CHECK:
    exists = path_value.exists()
    kind   = "DIR " if path_value.is_dir() else "FILE"
    status = "OK     " if exists else "MISSING"
    print(f"  [{kind}] {status}  {path_name}: {path_value}")
    if path_name in {"EXPORT_ROOT", "IMAGES_DIR", "LABELS_DIR",
                     "OBJECT_SPLIT_CSV", "BASE_DATA_YAML"} and not exists:
        all_paths_ok = False

# --- Load object_split.csv and print dataset stats ---
objectsplitdf = None
imagepaths    = list(IMAGES_DIR.glob("*")) if IMAGES_DIR.exists() else []

if OBJECT_SPLIT_CSV.exists():
    objectsplitdf = pd.read_csv(OBJECT_SPLIT_CSV, dtype={"object_id": str})
    objectsplitdf.columns = [c.strip() for c in objectsplitdf.columns]

    print("\nobject_split.csv columns:", list(objectsplitdf.columns))

    uniqueobjects = objectsplitdf.drop_duplicates(subset=["material", "object_id"])
    print("\nDataset stats")
    print("  Distinct objects:  ", len(uniqueobjects))
    print("  Distinct materials:", uniqueobjects["material"].nunique())
    print("  Total images:      ", len(imagepaths))
    display(uniqueobjects.groupby("material")["object_id"].nunique().rename("n_objects"))
else:
    raise FileNotFoundError(f"object_split.csv not found: {OBJECT_SPLIT_CSV}")

# --- Model instantiation smoke test ---
# Loads the first model to catch missing weights or a broken Ultralytics install.
try:
    YOLO(YOLO_MODELS[0])
    print(f"\nModel instantiation OK: {YOLO_MODELS[0]}")
except Exception as exc:
    raise RuntimeError(f"Model instantiation failed for {YOLO_MODELS[0]}") from exc

if not all_paths_ok:
    raise RuntimeError("One or more required paths are missing. Fix config before continuing.")

print("\nSanity check PASS")

In [ ]:
# === Cell 6: Filename Parsing ===

# Expected filename pattern: HDPE_obj_01_per_4_2026-04-21.png

FILENAME_REGEX = re.compile(
    r"^(?P<material>[A-Za-z0-9]+)"
    r"_obj_(?P<objectid>\d+)"
    r"_per_(?P<perspectiveid>\d+)"
    r"_(?P<date>\d{4}-\d{2}-\d{2})"
    r"\.(?P<extension>png|jpg|jpeg)$",
    re.IGNORECASE,
)

def parse_material_object_perspective(filename: str):
    """Parse 'HDPE_obj_01_per_4_2026-04-21.png' into (material, objectid, perspectiveid, datestr)."""
    match = FILENAME_REGEX.match(filename)
    if not match:
        raise ValueError(f"Filename does not match expected pattern: {filename}")
    return (match.group("material"),
            match.group("objectid"),
            match.group("perspectiveid"),
            match.group("date"))

# Map (material, objectid) -> image paths; check against object_split.csv.
object_to_images = defaultdict(list)
bad_filenames    = []

for image_path in imagepaths:
    filename = image_path.name
    try:
        material, objectid, perspectiveid, datestr = parse_material_object_perspective(filename)
        object_to_images[(material, objectid)].append(image_path)
    except ValueError:
        bad_filenames.append(filename)

print("Mapped objects from filenames:", len(object_to_images))

if bad_filenames:
    print(f"\nWARNING: {len(bad_filenames)} filenames did not match expected pattern:")
    for bad_name in bad_filenames[:10]:
        print("  -", bad_name)
    if len(bad_filenames) > 10:
        print(f"  ... and {len(bad_filenames) - 10} more")

if objectsplitdf is not None:
    uniqueobjects  = objectsplitdf.drop_duplicates(subset=["material", "object_id"])
    split_objects  = set(zip(uniqueobjects["material"], uniqueobjects["object_id"]))
    mapped_objects = set(object_to_images.keys())

    missing_in_images = split_objects - mapped_objects
    extra_in_images   = mapped_objects - split_objects

    print("\nConsistency checks")
    print("  Objects in object_split.csv but missing in images:", len(missing_in_images))
    print("  Objects in images but missing in object_split.csv:", len(extra_in_images))

    if missing_in_images:
        print("  Example missing:", list(missing_in_images)[:10])
    if extra_in_images:
        print("  Example extra:  ", list(extra_in_images)[:10])

    if len(missing_in_images) > 0:
        print("\nWARNING: some objects in the split CSV have no images — check IMAGES_DIR path.")
    else:
        print("\nFilename mapping OK — all split objects have images.")

In [ ]:
# === Cell 7: CV Fold Builder ===

random.seed(RANDOM_SEED)

cvdf      = objectsplitdf[objectsplitdf["split"] == "cv"].copy()
testdf    = objectsplitdf[objectsplitdf["split"] == "test"].copy()
skippeddf = objectsplitdf[objectsplitdf["split"] == "skipped"].copy()

def objects_to_image_paths(object_keys):
    """Convert iterable of (material, object_id) tuples to a sorted list of image paths."""
    paths = []
    for material, object_id in object_keys:
        images = object_to_images.get((material, object_id), [])
        if not images:
            print(f"WARNING: no images found for object {(material, object_id)}")
        paths.extend(images)
    return sorted(paths)

def build_split_summary_df(df, split_name):
    """Return a per-material summary dataframe with TOTAL row appended."""
    rows = []
    if df.empty:
        return pd.DataFrame(columns=["split", "material", "objects", "images"])
    for material in sorted(df["material"].unique()):
        material_df   = df[df["material"] == material]
        material_keys = list(zip(material_df["material"], material_df["object_id"]))
        rows.append({
            "split":    split_name,
            "material": material,
            "objects":  material_df["object_id"].nunique(),
            "images":   len(objects_to_image_paths(material_keys)),
        })
    summary_df = pd.DataFrame(rows)
    total_row  = pd.DataFrame([{
        "split":    split_name,
        "material": "TOTAL",
        "objects":  summary_df["objects"].sum(),
        "images":   summary_df["images"].sum(),
    }])
    return pd.concat([summary_df, total_row], ignore_index=True)

print("Split summary by material")
split_summary_df = pd.concat([
    build_split_summary_df(cvdf,      "CV"),
    build_split_summary_df(testdf,    "Test"),
    build_split_summary_df(skippeddf, "Skipped"),
], ignore_index=True)
display(split_summary_df)

# --- Fixed test set (used in every fold) ---
test_object_keys = list(zip(testdf["material"], testdf["object_id"]))
test_images      = objects_to_image_paths(test_object_keys)

TESTTXT = SPLITS_DIR / "test.txt"
with TESTTXT.open("w", encoding="utf-8") as f:
    for image_path in test_images:
        f.write(str(image_path.resolve()) + "\n")

# --- Object-level stratified K-fold per material ---
def make_folds_for_material(objectids, num_folds=NUM_FOLDS):
    """Shuffle objects and split as evenly as possible across num_folds."""
    objectids = list(objectids)
    random.shuffle(objectids)

    num_total = len(objectids)
    if num_total == 0:
        return [{"train": [], "val": []} for _ in range(num_folds)]
    if num_total < 2:
        print(f"WARNING: Material has only {num_total} CV object; no meaningful train/val split possible.")

    fold_sizes = [num_total // num_folds] * num_folds
    for i in range(num_total % num_folds):
        fold_sizes[i] += 1

    partitions, start = [], 0
    for fold_size in fold_sizes:
        end = start + fold_size
        partitions.append(objectids[start:end])
        start = end

    fold_defs = []
    for fold_index in range(num_folds):
        val_ids   = partitions[fold_index]
        train_ids = [obj for i, part in enumerate(partitions) if i != fold_index for obj in part]
        fold_defs.append({"train": train_ids, "val": val_ids})
    return fold_defs

materials          = sorted(cvdf["material"].unique())
folds              = defaultdict(dict)
material_fold_rows = []

for material in materials:
    material_cv_df     = cvdf[cvdf["material"] == material]
    material_objectids = sorted(material_cv_df["object_id"].unique())
    material_folds     = make_folds_for_material(material_objectids, num_folds=NUM_FOLDS)

    for fold_index, fold_def in enumerate(material_folds):
        train_keys_local = [(material, oid) for oid in fold_def["train"]]
        val_keys_local   = [(material, oid) for oid in fold_def["val"]]

        material_fold_rows.append({
            "fold":         f"fold_{fold_index + 1}",
            "material":     material,
            "trainobjects": len(fold_def["train"]),
            "trainimages":  len(objects_to_image_paths(train_keys_local)),
            "valobjects":   len(fold_def["val"]),
            "valimages":    len(objects_to_image_paths(val_keys_local)),
        })

        folds[fold_index][material] = fold_def

print("Per-material fold summary")
display(pd.DataFrame(material_fold_rows))

# Write per-fold train/val lists and data_fold_N.yaml files.
with open(BASE_DATA_YAML, "r", encoding="utf-8") as f:
    base_data = yaml.safe_load(f)

FOLD_CONFIGS      = {}
fold_summary_rows = []

for fold_index in range(NUM_FOLDS):
    fold_name = f"fold_{fold_index + 1}"
    fold_dir  = SPLITS_DIR / fold_name
    fold_dir.mkdir(parents=True, exist_ok=True)

    train_keys, val_keys = [], []
    for material in materials:
        train_keys.extend((material, oid) for oid in folds[fold_index][material]["train"])
        val_keys.extend((material, oid)   for oid in folds[fold_index][material]["val"])

    train_images = objects_to_image_paths(train_keys)
    val_images   = objects_to_image_paths(val_keys)

    train_txt = fold_dir / "train.txt"
    val_txt   = fold_dir / "val.txt"

    with train_txt.open("w", encoding="utf-8") as f:
        for image_path in train_images:
            f.write(str(image_path.resolve()) + "\n")

    with val_txt.open("w", encoding="utf-8") as f:
        for image_path in val_images:
            f.write(str(image_path.resolve()) + "\n")

    fold_data          = dict(base_data)
    fold_data["path"]  = str(EXPORT_ROOT.resolve())
    fold_data["train"] = str(train_txt.resolve())
    fold_data["val"]   = str(val_txt.resolve())
    fold_data["test"]  = str(TESTTXT.resolve())

    fold_yaml = SPLITS_DIR / f"data_{fold_name}.yaml"
    with fold_yaml.open("w", encoding="utf-8") as f:
        yaml.safe_dump(fold_data, f, sort_keys=False)

    FOLD_CONFIGS[fold_index] = {
        "name":            fold_name,
        "train_txt":       train_txt,
        "val_txt":         val_txt,
        "test_txt":        TESTTXT,
        "data_yaml":       fold_yaml,
        "num_train_images": len(train_images),
        "num_val_images":  len(val_images),
        "num_test_images": len(test_images),
        "num_train_objects": len(train_keys),
        "num_val_objects": len(val_keys),
    }

    fold_summary_rows.append({
        "fold":         fold_name,
        "trainobjects": len(train_keys),
        "trainimages":  len(train_images),
        "valobjects":   len(val_keys),
        "valimages":    len(val_images),
        "testobjects":  len(test_object_keys),
        "testimages":   len(test_images),
        "datayaml":     str(fold_yaml),
    })

print("Fold summary")
display(pd.DataFrame(fold_summary_rows))

# --- Test-set data.yaml for Inference (train/val/test all = TESTTXT) ---
TEST_DATA_YAML = INFERENCE_DIR / "data_test.yaml"
test_data          = dict(base_data)
test_data["path"]  = str(EXPORT_ROOT.resolve())
test_data["train"] = str(TESTTXT.resolve())
test_data["val"]   = str(TESTTXT.resolve())
test_data["test"]  = str(TESTTXT.resolve())
with TEST_DATA_YAML.open("w", encoding="utf-8") as f:
    yaml.safe_dump(test_data, f, sort_keys=False)

print("\nTEST_DATA_YAML:", TEST_DATA_YAML)
print("Finished writing splits and fold data.yaml files.")

In [ ]:
# === Cell 8: Training ===

# --- Helpers: run keys, serialization, deduplication ---
def make_run_key(model_base, fold_idx):
    """Stable identity for a training target (model x fold)."""
    return f"{model_base}__fold_{int(fold_idx)}"

def parse_model_version_size(model_name: str):
    """Extract (model_version, model_size) from filenames like 'yolo26n-seg.pt'."""
    stem = Path(model_name).stem            # e.g. "yolo26n-seg"
    base = stem.split("-")[0]               # e.g. "yolo26n"
    if not base.startswith("yolo") or len(base) < 6:
        return base, None
    return base[:-1], base[-1]              # ("yolo26", "n")

def get_augmentation_kwargs(profile_name: str):
    """Return augmentation kwargs for AUG_PROFILE_NAME from AUGMENTATION_PROFILES."""
    profile = AUGMENTATION_PROFILES.get(profile_name, {})
    return dict(profile)

def serialize_trained_runs(trained_runs):
    """Convert TRAINED_RUNS entries to JSON-friendly dicts."""
    out = []
    for r in trained_runs:
        out.append({
            "model_name":           str(r.get("model_name")) if r.get("model_name") is not None else None,
            "model_base":           str(r.get("model_base")),
            "model_version":        r.get("model_version"),
            "model_size":           r.get("model_size"),
            "fold_idx":             int(r.get("fold_idx")),
            "fold_name":            str(r.get("fold_name")),
            "data_yaml":            str(r.get("data_yaml")) if r.get("data_yaml") is not None else None,
            "run_dir":              str(r.get("run_dir")) if r.get("run_dir") is not None else None,
            "experiment_name":      r.get("experiment_name"),
            "num_folds":            r.get("num_folds"),
            "train_epochs":         r.get("train_epochs"),
            "train_patience":       r.get("train_patience"),
            "batch_size":           r.get("batch_size"),
            "image_size":           r.get("image_size"),
            "image_rect":           r.get("image_rect"),
            "train_lr0":            r.get("train_lr0"),
            "augmentation_profile": r.get("augmentation_profile"),
            "device":               r.get("device"),
            "optimizer":            r.get("optimizer"),
        })
    return out

def deserialize_trained_runs(payload_runs):
    """Inverse of serialize_trained_runs — restore Path objects."""
    restored = []
    for r in payload_runs:
        restored.append({
            "model_name":           r.get("model_name"),
            "model_base":           r.get("model_base"),
            "model_version":        r.get("model_version"),
            "model_size":           r.get("model_size"),
            "fold_idx":             int(r.get("fold_idx")),
            "fold_name":            r.get("fold_name"),
            "data_yaml":            Path(r["data_yaml"]) if r.get("data_yaml") is not None else None,
            "run_dir":              Path(r["run_dir"])   if r.get("run_dir")   is not None else None,
            "experiment_name":      r.get("experiment_name"),
            "num_folds":            r.get("num_folds"),
            "train_epochs":         r.get("train_epochs"),
            "train_patience":       r.get("train_patience"),
            "batch_size":           r.get("batch_size"),
            "image_size":           r.get("image_size"),
            "image_rect":           r.get("image_rect"),
            "train_lr0":            r.get("train_lr0"),
            "augmentation_profile": r.get("augmentation_profile"),
            "device":               r.get("device"),
            "optimizer":            r.get("optimizer"),
        })
    return restored

def deduplicate_trained_runs(trained_runs):
    """Keep the latest entry per (model_base, fold_idx)."""
    dedup = {}
    for r in trained_runs:
        dedup[make_run_key(r["model_base"], r["fold_idx"])] = r
    return list(dedup.values())

def save_trained_runs_metadata(trained_runs, trained_runs_file):
    """Serialize and persist TRAINED_RUNS to a JSON file."""
    trained_runs = deduplicate_trained_runs(trained_runs)
    payload = {
        "experiment_name":      EXPERIMENT_NAME,
        "created_at":           datetime.now().isoformat(),
        "image_size":           IMAGE_SIZE,
        "batch_size":           BATCH_SIZE,
        "image_rect":           IMAGE_RECT,
        "train_epochs":         TRAIN_EPOCHS,
        "train_patience":       TRAIN_PATIENCE,
        "num_folds":            NUM_FOLDS,
        "train_lr0":            TRAIN_LR0,
        "augmentation_profile": AUG_PROFILE_NAME,
        "device":               str(DEVICE),
        "optimizer":            "SGD",
        "runs":                 serialize_trained_runs(trained_runs),
    }
    trained_runs_file.parent.mkdir(parents=True, exist_ok=True)
    with open(trained_runs_file, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    return trained_runs

# --- Metadata file + experiment config CSV ---
RUNS_SAVE_DIR     = RUNS_DIR / "notebook_metadata"
trained_runs_file = RUNS_SAVE_DIR / f"{EXPERIMENT_NAME}_trained_runs.json"
CONFIG_CSV_PATH   = RESULTS_DIR / f"{PROJECT_NAME}_train_config.csv"

def save_experiment_config_csv(config_csv_path: Path):
    """Save a one-row CSV with the experiment-level configuration."""
    config_row = {
        "experiment_name":             EXPERIMENT_NAME,
        "export_root":                 str(EXPORT_ROOT.resolve()),
        "images_dir":                  str(IMAGES_DIR.resolve()),
        "labels_dir":                  str(LABELS_DIR.resolve()),
        "object_split_csv":            str(OBJECT_SPLIT_CSV.resolve()),
        "metadata_agg_csv":            str(METADATA_AGG_CSV.resolve()),
        "base_data_yaml":              str(BASE_DATA_YAML.resolve()),
        "train_root":                  str(TRAIN_ROOT.resolve()),
        "splits_dir":                  str(SPLITS_DIR.resolve()),
        "runs_dir":                    str(RUNS_DIR.resolve()),
        "results_dir":                 str(RESULTS_DIR.resolve()),
        "inference_dir":               str(INFERENCE_DIR.resolve()),
        "random_seed":                 RANDOM_SEED,
        "num_folds":                   NUM_FOLDS,
        "folds_to_run":                str(FOLDS_TO_RUN),
        "yolo_models":                 ";".join(YOLO_MODELS),
        "train_epochs":                TRAIN_EPOCHS,
        "train_patience":              TRAIN_PATIENCE,
        "batch_size":                  BATCH_SIZE,
        "image_size":                  IMAGE_SIZE,
        "image_rect":                  IMAGE_RECT,
        "train_lr0":                   TRAIN_LR0,
        "augmentation_profile":        AUG_PROFILE_NAME,
        "best_epoch_metric":           BEST_EPOCH_METRIC,
        "best_model_selection_metric": BEST_MODEL_SELECTION_METRIC,
        "f1_metric_name":              F1_METRIC_NAME,
        "device":                      str(DEVICE),
        "workers":                     WORKERS,
        "optimizer":                   "SGD",
    }
    config_csv_path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([config_row]).to_csv(config_csv_path, index=False)
    print(f"Saved train config CSV to: {config_csv_path}")

# --- Load existing run metadata (already trained runs are skipped) ---
if trained_runs_file.exists():
    with open(trained_runs_file, "r", encoding="utf-8") as f:
        old_payload = json.load(f)
    TRAINED_RUNS = deserialize_trained_runs(old_payload.get("runs", []))
    TRAINED_RUNS = deduplicate_trained_runs(TRAINED_RUNS)
    print(f"Loaded existing TRAINED_RUNS metadata from: {trained_runs_file}")
    print(f"Existing saved runs: {len(TRAINED_RUNS)}")
else:
    TRAINED_RUNS = []
    print(f"No existing metadata file found. Starting fresh at: {trained_runs_file}")

existing_run_keys = {make_run_key(r["model_base"], r["fold_idx"]) for r in TRAINED_RUNS}

save_experiment_config_csv(CONFIG_CSV_PATH)

# --- Determine folds to run ---
if FOLDS_TO_RUN == "all":
    folds_to_run = sorted(FOLD_CONFIGS.keys())
else:
    folds_to_run = sorted(int(f) for f in FOLDS_TO_RUN)

print("Folds to run: ", folds_to_run)
print("Models to run:", YOLO_MODELS)

total_runs       = len(YOLO_MODELS) * len(folds_to_run)
run_counter      = 0
trained_now      = 0
skipped_existing = 0
print(f"Planned training targets: {total_runs}")

# --- Main training loop ---
for model_name in tqdm(YOLO_MODELS, desc="Models", position=0):
    model_base = Path(model_name).stem
    model_version, model_size = parse_model_version_size(model_name)

    for fold_idx in tqdm(folds_to_run, desc=f"Folds for {model_base}", position=1, leave=False):
        fold_conf = FOLD_CONFIGS[fold_idx]
        fold_name = fold_conf["name"]
        data_yaml = fold_conf["data_yaml"]

        run_name = f"{EXPERIMENT_NAME}_{model_base}_{fold_name}"
        run_key  = make_run_key(model_base, fold_idx)

        run_counter += 1
        print(f"\n=== Target {run_counter}/{total_runs}: model={model_base}, fold={fold_name} ===")
        print(f"  data:           {data_yaml}")
        print(f"  run name:       {run_name}")
        print(f"  DEVICE:         {DEVICE}")
        print(f"  model_version:  {model_version}, model_size: {model_size}")
        print(f"  train_lr0:      {TRAIN_LR0}, augmentation_profile: {AUG_PROFILE_NAME}")

        if run_key in existing_run_keys:
            skipped_existing += 1
            print(f"  -> SKIP: already present in metadata ({run_key})")
            continue

        yolo_model = YOLO(model_name)
        aug_kwargs = get_augmentation_kwargs(AUG_PROFILE_NAME)

        # optimizer="SGD" is required; otherwise Ultralytics auto-selects AdamW and
        # ignores lr0/momentum, which does not match the SGD-based HPO and final training.
        results = yolo_model.train(
            data=str(data_yaml),
            epochs=TRAIN_EPOCHS,
            patience=TRAIN_PATIENCE,
            batch=BATCH_SIZE,           # fixed; see BATCH_SIZE in Config
            imgsz=IMAGE_SIZE,
            device=DEVICE,
            project=str(RUNS_DIR),
            name=run_name,
            exist_ok=True,
            task=TASK,
            workers=WORKERS,
            rect=IMAGE_RECT,
            lr0=TRAIN_LR0,
            optimizer="SGD",            # required, see above
            seed=RANDOM_SEED,
            deterministic=True,
            verbose=False,
            **aug_kwargs,
        )

        # Run directory as written by Ultralytics
        if results is not None and hasattr(results, "save_dir"):
            run_dir = Path(results.save_dir)
        else:
            run_dir = RUNS_DIR / run_name
            print(f"WARNING: YOLO.train() returned None; inferring run_dir as: {run_dir}")

        print(f"  -> run dir: {run_dir}")

        new_entry = {
            "model_name":           model_name,
            "model_base":           model_base,
            "model_version":        model_version,
            "model_size":           model_size,
            "fold_idx":             fold_idx,
            "fold_name":            fold_name,
            "data_yaml":            Path(data_yaml),
            "run_dir":              run_dir,
            "experiment_name":      EXPERIMENT_NAME,
            "num_folds":            NUM_FOLDS,
            "train_epochs":         TRAIN_EPOCHS,
            "train_patience":       TRAIN_PATIENCE,
            "batch_size":           BATCH_SIZE,
            "image_size":           IMAGE_SIZE,
            "image_rect":           IMAGE_RECT,
            "train_lr0":            TRAIN_LR0,
            "augmentation_profile": AUG_PROFILE_NAME,
            "device":               str(DEVICE),
            "optimizer":            "SGD",
        }
        TRAINED_RUNS.append(new_entry)
        TRAINED_RUNS = deduplicate_trained_runs(TRAINED_RUNS)
        existing_run_keys = {make_run_key(r["model_base"], r["fold_idx"]) for r in TRAINED_RUNS}
        TRAINED_RUNS = save_trained_runs_metadata(TRAINED_RUNS, trained_runs_file)
        trained_now += 1

        print(f"  -> metadata saved to: {trained_runs_file}")
        print(f"  -> total saved runs now: {len(TRAINED_RUNS)}")

print("\nTraining loop finished.")
print(f"Newly trained this execution:        {trained_now}")
print(f"Skipped because already in metadata: {skipped_existing}")
print(f"Total runs currently tracked:        {len(TRAINED_RUNS)}")
print(f"Metadata file: {trained_runs_file}")
print(f"Config CSV:    {CONFIG_CSV_PATH}")

for r in sorted(TRAINED_RUNS, key=lambda x: (x["model_base"], x["fold_idx"])):
    print(
        f"  {r['model_base']} / {r['fold_name']} "
        f"(version={r.get('model_version')}, size={r.get('model_size')}, "
        f"lr0={r.get('train_lr0')}, opt={r.get('optimizer')}, aug={r.get('augmentation_profile')}) "
        f"-> {r['run_dir']}"
    )

In [ ]:
# === Cell 9: Inference ===

# --- Helpers ---
# Ultralytics metric attributes are methods or plain arrays depending on the version.
def _safe_get(obj, *attrs, default=np.nan):
    """Walk obj.attr1.attr2... safely; call any callable encountered."""
    try:
        cur = obj
        for a in attrs:
            cur = getattr(cur, a)
            if callable(cur) and not isinstance(cur, np.ndarray):
                cur = cur()
        return cur
    except Exception:
        return default

def _safe_float(obj, *attrs, default=np.nan):
    """_safe_get coerced to float."""
    val = _safe_get(obj, *attrs, default=default)
    try:
        return float(val)
    except Exception:
        return default

def _find_col(df, patterns):
    """First column whose name contains any pattern (case-insensitive)."""
    for p in patterns:
        for c in df.columns:
            if p.lower() in c.lower():
                return c
    return None

def read_non_empty_lines(path):
    """Return non-empty stripped lines from a text file."""
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

# --- Test image set ---
TEST_IMAGE_PATHS = [Path(p) for p in read_non_empty_lines(TESTTXT)]
if len(TEST_IMAGE_PATHS) == 0:
    raise RuntimeError(f"TESTTXT exists but contains no test images: {TESTTXT}")

print(f"Test data yaml:    {TEST_DATA_YAML}")
print(f"Number of test images: {len(TEST_IMAGE_PATHS)}")
print(f"CONF_THRESHOLD = {CONF_THRESHOLD}, IOU_THRESHOLD = {IOU_THRESHOLD}")

# --- Per-run test evaluation ---
if "TRAINED_RUNS" not in globals():
    raise RuntimeError("TRAINED_RUNS is not defined. Make sure the Training cell has run.")

RUN_TEST_METRICS_CSV   = RESULTS_DIR / f"{PROJECT_NAME}_run_metrics_inference.csv"
MODEL_TEST_METRICS_CSV = RESULTS_DIR / f"{PROJECT_NAME}_model_metrics_inference.csv"
PER_CLASS_METRICS_CSV  = RESULTS_DIR / f"{PROJECT_NAME}_per_class_metrics_inference.csv"

test_metrics_rows = []
per_class_rows    = []

for run in sorted(TRAINED_RUNS, key=lambda r: (r["model_base"], r["fold_idx"])):
    model_name = run["model_name"]
    model_base = run["model_base"]
    fold_idx   = run["fold_idx"]
    fold_name  = run["fold_name"]
    run_dir    = Path(run["run_dir"])

    weights_path = run_dir / "weights" / "best.pt"
    if not weights_path.exists():
        print(f"WARNING: best weights not found for {model_base}, fold={fold_name}: {weights_path}")
        continue

    model_version, model_size = parse_model_version_size(model_base)

    print("=" * 100)
    print(f"Model base    : {model_base}")
    print(f"Model version : {model_version}, size={model_size}")
    print(f"Fold          : {fold_name} (idx={fold_idx})")
    print(f"Weights       : {weights_path}")
    print(f"Test YAML     : {TEST_DATA_YAML}")
    print(f"Num test imgs : {len(TEST_IMAGE_PATHS)}")

    model = YOLO(str(weights_path))

    # Write val() outputs to a per-run folder under RUNS_DIR.
    val_name = f"test_eval__{model_base}__{fold_name}"

    val_results = model.val(
        data=str(TEST_DATA_YAML),
        split="test",
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        device=DEVICE,
        workers=WORKERS,
        rect=IMAGE_RECT,
        plots=False,
        save_json=False,
        verbose=False,
        project=str(RUNS_DIR),    # keeps val outputs out of the cwd
        name=val_name,
        exist_ok=True,
    )

    # --- Aggregate mask metrics ---
    seg   = _safe_get(val_results, "seg", default=None)
    prec  = _safe_float(seg, "mp")
    rec   = _safe_float(seg, "mr")
    map50 = _safe_float(seg, "map50")
    map_  = _safe_float(seg, "map")
    f1    = (2 * prec * rec / (prec + rec)) if (not np.isnan(prec + rec) and (prec + rec) > 0) else np.nan

    speed_raw = getattr(val_results, "speed", {}) or {}

    def _spd(speed_dict, key_patterns):
        for k in speed_dict:
            for p in key_patterns:
                if p.lower() in k.lower():
                    return float(speed_dict[k])
        return np.nan

    row = {
        "infer_precision(M)":   prec,
        "infer_recall(M)":      rec,
        "infer_mAP50(M)":       map50,
        "infer_mAP50-95(M)":    map_,
        "infer_F1(M)":          f1,
        "speed_preprocess_ms":  _spd(speed_raw, ["preprocess"]),
        "speed_inference_ms":   _spd(speed_raw, ["inference"]),
        "speed_postprocess_ms": _spd(speed_raw, ["postprocess"]),
    }
    row["speed_total_ms"] = float(np.nansum([row["speed_preprocess_ms"],
                                             row["speed_inference_ms"],
                                             row["speed_postprocess_ms"]]))
    row["fps"] = (1000.0 / row["speed_total_ms"]) if row["speed_total_ms"] > 0 else np.nan

    row.update({
        "model_name":      model_name,
        "model_base":      model_base,
        "model_version":   model_version,
        "model_size":      model_size,
        "fold_idx":        int(fold_idx),
        "fold_name":       fold_name,
        "run_dir":         str(run_dir),
        "weights_path":    str(weights_path),
        "num_test_images": len(TEST_IMAGE_PATHS),
        "test_data_yaml":  str(TEST_DATA_YAML),
    })
    test_metrics_rows.append(row)

    # --- Per-class metrics ---
    # seg.all_ap is a plain ndarray [n_classes, n_iou_thresholds] (Ultralytics 8.4.x);
    # it is read directly because ap50()/ap() are not callable on it.
    try:
        names_map    = val_results.names                                                   # {0: "body", ...}
        ap_class_idx = np.asarray(_safe_get(seg, "ap_class_index"))                        # [n_classes]
        all_ap       = np.asarray(_safe_get(seg, "all_ap"))                                # [n_classes, n_iou]

        if all_ap.ndim == 2 and len(ap_class_idx) > 0:
            ap50_pc   = all_ap[:, 0]            # AP @ IoU=0.50
            ap5095_pc = all_ap.mean(axis=1)     # mean AP @ IoU=0.50:0.95
            p_pc = np.asarray(_safe_get(seg, "p", default=np.full(len(ap_class_idx), np.nan)))
            r_pc = np.asarray(_safe_get(seg, "r", default=np.full(len(ap_class_idx), np.nan)))

            for j, cls_idx in enumerate(ap_class_idx):
                cls_name = names_map.get(int(cls_idx), str(cls_idx))
                p_j  = float(p_pc[j]) if j < len(p_pc) else np.nan
                r_j  = float(r_pc[j]) if j < len(r_pc) else np.nan
                f1_j = (2 * p_j * r_j / (p_j + r_j)) if (not np.isnan(p_j + r_j) and (p_j + r_j) > 0) else np.nan
                per_class_rows.append({
                    "model_name":  model_name,
                    "model_base":  model_base,
                    "fold_idx":    int(fold_idx),
                    "fold_name":   fold_name,
                    "class_id":    int(cls_idx),
                    "class":       cls_name,
                    "AP50":        round(float(ap50_pc[j]),   6),
                    "AP50-95":     round(float(ap5095_pc[j]), 6),
                    "precision":   round(p_j,  6),
                    "recall":      round(r_j,  6),
                    "F1":          round(f1_j, 6) if not np.isnan(f1_j) else np.nan,
                })
    except Exception as e:
        print(f"  WARNING: per-class metrics not extracted for {model_base}/{fold_name}: {e}")

# --- Save run-level and model-level inference metrics CSVs ---
if test_metrics_rows:
    run_test_metrics_df = pd.DataFrame(test_metrics_rows)
    run_test_metrics_df.to_csv(RUN_TEST_METRICS_CSV, index=False)
    print("\nPer-run inference metrics (first rows):")
    display(run_test_metrics_df.head())
    print(f"Saved per-run inference metrics CSV to: {RUN_TEST_METRICS_CSV}")

    agg_cols = {
        "infer_mAP50(M)":       ["mean", "std"],
        "infer_mAP50-95(M)":    ["mean", "std"],
        "infer_precision(M)":   ["mean", "std"],
        "infer_recall(M)":      ["mean", "std"],
        "infer_F1(M)":          ["mean", "std"],
        "speed_preprocess_ms":  ["mean", "std"],
        "speed_inference_ms":   ["mean", "std"],
        "speed_postprocess_ms": ["mean", "std"],
        "speed_total_ms":       ["mean", "std"],
        "fps":                  ["mean", "std"],
    }
    model_metrics = (
        run_test_metrics_df
        .groupby(["model_base", "model_version", "model_size"])
        .agg(agg_cols)
    )
    model_metrics.columns = [f"{col[0]}__{col[1]}" for col in model_metrics.columns.to_flat_index()]
    model_metrics = model_metrics.reset_index()
    model_metrics.to_csv(MODEL_TEST_METRICS_CSV, index=False)

    print("\nPer-model (architecture) aggregated inference metrics:")
    display(model_metrics)
    print(f"Saved model aggregated inference metrics CSV to: {MODEL_TEST_METRICS_CSV}")

    # --- Per-class breakdown (body / cap / label) across runs ---
    if per_class_rows:
        per_class_df = pd.DataFrame(per_class_rows)
        per_class_df.to_csv(PER_CLASS_METRICS_CSV, index=False)
        print("\nPer-class test metrics (first rows):")
        display(per_class_df.head(12))
        print(f"Saved per-class metrics CSV to: {PER_CLASS_METRICS_CSV}")

        # Mean/std per (model_base, class) over folds
        per_class_summary = (
            per_class_df
            .groupby(["model_base", "class_id", "class"])
            .agg(AP50_mean=("AP50", "mean"),     AP50_std=("AP50", "std"),
                 AP5095_mean=("AP50-95", "mean"), AP5095_std=("AP50-95", "std"),
                 F1_mean=("F1", "mean"),         F1_std=("F1", "std"),
                 n_runs=("AP50", "count"))
            .reset_index()
        )
        print("\nPer-class summary (mean over folds):")
        display(per_class_summary.round(4))
    else:
        print("\nNo per-class metrics were extracted.")
else:
    print("No inference metrics were collected (no runs had best weights).")

In [ ]:
# === Cell 10: Metrics and Plots ===

# --- Best-epoch validation metrics per CV run ---
CANDIDATE_METRICS = [
    "metrics/mAP50(M)",
    "metrics/mAP50-95(M)",
    "metrics/precision(M)",
    "metrics/recall(M)",
]

all_best_rows = []
missing_runs  = []

for run in TRAINED_RUNS:
    run_dir     = Path(run["run_dir"])
    results_csv = run_dir / "results.csv"
    if not results_csv.exists():
        missing_runs.append(str(results_csv))
        continue

    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]

    available_metrics = [m for m in CANDIDATE_METRICS if m in df.columns]
    if not available_metrics:
        print(f"No expected metric columns found in {results_csv}")
        continue

    selection_metric = BEST_EPOCH_METRIC if BEST_EPOCH_METRIC in df.columns else available_metrics[0]
    best_idx = pd.to_numeric(df[selection_metric], errors="coerce").idxmax()
    best_row = df.loc[best_idx].copy()

    row_dict = {
        "model_name":             run["model_name"],
        "model_base":              run["model_base"],
        "model_version":          run.get("model_version"),
        "model_size":              run.get("model_size"),
        "fold_idx":                run["fold_idx"],
        "fold_name":               run["fold_name"],
        "run_dir":                 str(run_dir),
        "best_epoch":              int(best_row["epoch"]) if "epoch" in df.columns else int(best_idx),
        "selection_metric":        selection_metric,
        "selection_metric_value":  float(best_row[selection_metric]),
    }
    for m in available_metrics:
        row_dict[m] = float(best_row[m])

    if ("metrics/precision(M)" in df.columns) and ("metrics/recall(M)" in df.columns):
        p = float(best_row["metrics/precision(M)"])
        r = float(best_row["metrics/recall(M)"])
        row_dict[F1_METRIC_NAME] = 0.0 if (p + r) == 0 else 2 * (p * r) / (p + r)

    all_best_rows.append(row_dict)

if missing_runs:
    print(f"WARNING: {len(missing_runs)} runs had no results.csv (first 5):")
    for s in missing_runs[:5]:
        print("  -", s)

best_runs_df = pd.DataFrame(all_best_rows)
if best_runs_df.empty:
    raise RuntimeError("No valid results.csv files found from TRAINED_RUNS.")

# --- best_fold indicator per model_base ---
metric_cols = [c for c in CANDIDATE_METRICS if c in best_runs_df.columns]
if F1_METRIC_NAME in best_runs_df.columns:
    metric_cols.append(F1_METRIC_NAME)

if BEST_MODEL_SELECTION_METRIC in best_runs_df.columns:
    sel_metric = BEST_MODEL_SELECTION_METRIC
elif BEST_EPOCH_METRIC in best_runs_df.columns:
    sel_metric = BEST_EPOCH_METRIC
else:
    sel_metric = metric_cols[0]

best_fold_idx_per_model = best_runs_df.groupby("model_base")[sel_metric].idxmax()
best_runs_df["best_fold"] = False
best_runs_df.loc[best_fold_idx_per_model, "best_fold"] = True

# --- Display tables ---
display_cols = (
    ["model_base", "fold_idx", "best_fold", "best_epoch",
     "selection_metric", "selection_metric_value"]
    + [c for c in best_runs_df.columns
       if c not in ["model_name", "model_base", "fold_idx", "fold_name",
                    "best_fold", "best_epoch", "selection_metric",
                    "selection_metric_value", "run_dir", "model_version", "model_size"]]
    + ["run_dir"]
)
best_runs_display = best_runs_df[display_cols].copy()

metric_rename_map = {}
for c in best_runs_display.columns:
    if c in metric_cols:
        metric_short = c.split("/")[-1]      # e.g. "mAP50(M)"
        metric_rename_map[c] = f"train_{metric_short}"
best_runs_display = best_runs_display.rename(columns=metric_rename_map)
best_runs_display = best_runs_display.sort_values(["model_base", "fold_idx"]).reset_index(drop=True)

print("Training metrics: per-run best epoch (validation)")
display(best_runs_display)

agg_dict = {m: ["mean", "std"] for m in metric_cols}
model_summary = best_runs_df.groupby("model_base").agg(agg_dict)
model_summary.columns = [f"{col[0]}__{col[1]}" for col in model_summary.columns]
model_summary = model_summary.reset_index()

summary_rename_map = {}
for old_col in model_summary.columns:
    for m in metric_cols:
        if old_col.startswith(m + "__"):
            metric_short = m.split("/")[-1]
            suffix       = old_col.split("__", 1)[1]
            summary_rename_map[old_col] = f"train_{metric_short}__{suffix}"
            break
model_summary_display = model_summary.rename(columns=summary_rename_map)

print("Training metrics: per-model aggregated (validation)")
display(model_summary_display)

# --- Color palette per model_base ---
sorted_models = sorted(model_summary_display["model_base"].unique())
n_colors      = len(sorted_models)
cmap_name     = "tab10" if n_colors <= 10 else "tab20"
colors_rgb    = plt.get_cmap(cmap_name)(np.linspace(0, 0.9, n_colors))[:, :3]

model_colors = {}
for model_base, col_rgb in zip(sorted_models, colors_rgb):
    model_colors[model_base] = tuple(col_rgb)

# --- 1. Multi-model training metric bars (mean ± std) ---
n_metrics = len(metric_cols)
fig_bar, axes = plt.subplots(1, n_metrics, figsize=(5 * n_metrics, 5), squeeze=False)
axes = axes[0]
fig_bar.suptitle(f"Per-model validation metrics — {PROJECT_NAME}", fontsize=12)

for ax, metric in zip(axes, metric_cols):
    metric_short = metric.split("/")[-1]
    mean_col     = f"train_{metric_short}__mean"
    std_col      = f"train_{metric_short}__std"
    for m_base in model_summary_display["model_base"]:
        row = model_summary_display[model_summary_display["model_base"] == m_base]
        if row.empty or mean_col not in row.columns:
            continue
        means = row[mean_col]
        stds  = row.get(std_col, pd.Series([0.0])).fillna(0)
        ax.bar(m_base, means, yerr=stds,
               color=model_colors[m_base], capsize=5,
               label=m_base if metric == metric_cols[0] else None)
    ax.set_title(f"{metric_short} (train)")
    ax.set_ylabel("score"); ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=45); ax.grid(axis="y", alpha=0.3)
    if metric == metric_cols[0]:
        ax.legend(fontsize=8)
plt.tight_layout()
bar_plot_path = RESULTS_DIR / f"{PROJECT_NAME}_bar_plots_training.png"
fig_bar.savefig(bar_plot_path, bbox_inches="tight", dpi=150)
plt.show()
print("Saved:", bar_plot_path)

# --- 2. Multi-model training fold-wise scatter ---
fig_scatter, axes = plt.subplots(1, n_metrics, figsize=(5 * n_metrics, 5), squeeze=False)
axes = axes[0]
fig_scatter.suptitle(f"Per-fold validation scores — {PROJECT_NAME}", fontsize=12)

for ax, metric in zip(axes, metric_cols):
    metric_short = metric.split("/")[-1]
    for i, model_base in enumerate(sorted_models):
        vals = best_runs_df.loc[best_runs_df["model_base"] == model_base, metric].dropna().values
        if len(vals) == 0:
            continue
        x = np.full(len(vals), i, dtype=float)
        jitter = np.random.uniform(-0.08, 0.08, size=len(vals))
        ax.scatter(x + jitter, vals, alpha=0.8, color=model_colors[model_base],
                   label=model_base if metric == metric_cols[0] else None)
        ax.hlines(float(np.mean(vals)), i - 0.2, i + 0.2,
                  linewidth=2, color=model_colors[model_base])
    ax.set_title(f"{metric_short} (train)"); ax.set_ylim(0, 1)
    ax.set_xticks(range(len(sorted_models)))
    ax.set_xticklabels(sorted_models, rotation=45)
    ax.set_ylabel("score"); ax.grid(axis="y", alpha=0.3)
if len(sorted_models) > 1:
    axes[0].legend(fontsize=8)
plt.tight_layout()
scatter_plot_path = RESULTS_DIR / f"{PROJECT_NAME}_scatter_plots_training.png"
fig_scatter.savefig(scatter_plot_path, bbox_inches="tight", dpi=150)
plt.show()
print("Saved:", scatter_plot_path)

best_runs_csv_path     = RESULTS_DIR / f"{PROJECT_NAME}_run_metrics_training.csv"
model_summary_csv_path = RESULTS_DIR / f"{PROJECT_NAME}_model_metrics_training.csv"
best_runs_display.to_csv(best_runs_csv_path,     index=False)
model_summary_display.to_csv(model_summary_csv_path, index=False)
print("Saved:", best_runs_csv_path)
print("Saved:", model_summary_csv_path)

# --- 3. Training curves (loss and metrics) of each model's best fold ---
loss_specs = [
    ("Train box loss", ["train/box", "box_loss"]),
    ("Train seg loss", ["train/seg", "seg_loss"]),
    ("Train cls loss", ["train/cls", "cls_loss"]),
    ("Train dfl loss", ["train/dfl", "dfl_loss"]),
    ("Val box loss",   ["val/box"]),
    ("Val seg loss",   ["val/seg"]),
    ("Val cls loss",   ["val/cls"]),
    ("Val dfl loss",   ["val/dfl"]),
]
metric_specs = [
    ("Precision", ["precision(M)", "precision"]),
    ("Recall",    ["recall(M)",    "recall"]),
    ("mAP@50",    ["mAP50(M)",     "mAP50"]),
    ("mAP@50-95", ["mAP50-95(M)", "mAP50-95"]),
]

best_per_model_runs = best_runs_df.loc[best_runs_df["best_fold"]]
for _, brun in best_per_model_runs.iterrows():
    model_base   = brun["model_base"]
    run_dir      = Path(brun["run_dir"])
    results_csv  = run_dir / "results.csv"
    if not results_csv.exists():
        continue
    tdf = pd.read_csv(results_csv)
    tdf.columns = [c.strip() for c in tdf.columns]
    ep_col = _find_col(tdf, ["epoch"]) or tdf.columns[0]
    ep = pd.to_numeric(tdf[ep_col], errors="coerce")

    # --- loss curves ---
    fig_l, axes_l = plt.subplots(2, 4, figsize=(18, 7))
    fig_l.suptitle(f"Loss curves — {model_base} ({brun['fold_name']})", fontsize=12)
    for ax, (lbl, pats) in zip(axes_l.reshape(-1), loss_specs):
        col = _find_col(tdf, pats)
        clr = "tab:blue" if "Train" in lbl else "tab:orange"
        if col:
            ax.plot(ep, pd.to_numeric(tdf[col], errors="coerce"), color=clr)
            ax.set_title(lbl, fontsize=9); ax.set_xlabel("Epoch", fontsize=8)
            ax.grid(alpha=0.3)
        else:
            ax.set_title(f"{lbl}\n(not found)", fontsize=9); ax.axis("off")
    plt.tight_layout()
    loss_png = RESULTS_DIR / f"{PROJECT_NAME}_loss_curves__{model_base}.png"
    fig_l.savefig(loss_png, bbox_inches="tight", dpi=150)
    plt.show()
    print("Saved:", loss_png)

    # --- val metric curves ---
    fig_m, axes_m = plt.subplots(1, 4, figsize=(18, 4))
    fig_m.suptitle(f"Validation metrics per epoch — {model_base} ({brun['fold_name']})", fontsize=12)
    for ax, (lbl, pats) in zip(axes_m, metric_specs):
        col = _find_col(tdf, pats)
        if col:
            vals = pd.to_numeric(tdf[col], errors="coerce")
            ax.plot(ep, vals, color="tab:green")
            best_i = vals.idxmax()
            bv, be = vals[best_i], int(ep[best_i])
            ax.axvline(be, color="red", linestyle="--", alpha=0.6,
                       label=f"best={bv:.3f} @ep{be}")
            ax.set_title(lbl); ax.set_xlabel("Epoch")
            ax.set_ylim(0, 1); ax.legend(fontsize=8); ax.grid(alpha=0.3)
        else:
            ax.set_title(f"{lbl}\n(not found)"); ax.axis("off")
    plt.tight_layout()
    metric_png = RESULTS_DIR / f"{PROJECT_NAME}_metric_curves__{model_base}.png"
    fig_m.savefig(metric_png, bbox_inches="tight", dpi=150)
    plt.show()
    print("Saved:", metric_png)

# --- 4. Multi-model inference speed plot (mean per model_base) ---
infer_run_csv = RESULTS_DIR / f"{PROJECT_NAME}_run_metrics_inference.csv"
if infer_run_csv.exists():
    infer_run_df = pd.read_csv(infer_run_csv)

    speed_agg = (
        infer_run_df
        .groupby("model_base")
        .agg(
            preprocess_ms = ("speed_preprocess_ms",  "mean"),
            inference_ms  = ("speed_inference_ms",   "mean"),
            postprocess_ms= ("speed_postprocess_ms", "mean"),
            fps           = ("fps",                  "mean"),
        )
        .reindex(sorted_models)
    )

    fig_spd, axes_spd = plt.subplots(1, 2, figsize=(13, 5))
    fig_spd.suptitle(f"Inference speed on test set — {PROJECT_NAME}", fontsize=12)

    # Stacked latency bars
    ax_bar = axes_spd[0]
    bottoms = np.zeros(len(speed_agg))
    stage_labels = ["preprocess_ms", "inference_ms", "postprocess_ms"]
    stage_colors = ["tab:blue", "tab:orange", "tab:green"]
    x_pos = np.arange(len(speed_agg))
    for stage, clr in zip(stage_labels, stage_colors):
        vals = speed_agg[stage].fillna(0.0).values
        ax_bar.bar(x_pos, vals, bottom=bottoms, color=clr, label=stage.replace("_ms", ""))
        bottoms = bottoms + vals
    for i, total in enumerate(bottoms):
        ax_bar.text(i, total + 0.2, f"{total:.2f} ms", ha="center", va="bottom", fontsize=9)
    ax_bar.set_xticks(x_pos)
    ax_bar.set_xticklabels(speed_agg.index, rotation=45, ha="right")
    ax_bar.set_ylabel("ms / image"); ax_bar.set_title("Latency breakdown (mean over folds)")
    ax_bar.legend(); ax_bar.grid(axis="y", alpha=0.3)

    # FPS bars
    ax_fps = axes_spd[1]
    fps_vals = speed_agg["fps"].fillna(0.0).values
    ax_fps.bar(x_pos, fps_vals,
               color=[model_colors.get(m, (0.5, 0.5, 0.5)) for m in speed_agg.index])
    for i, v in enumerate(fps_vals):
        ax_fps.text(i, v + 0.5, f"{v:.1f}", ha="center", va="bottom", fontsize=10)
    ax_fps.set_xticks(x_pos)
    ax_fps.set_xticklabels(speed_agg.index, rotation=45, ha="right")
    ax_fps.set_ylabel("Frames / second"); ax_fps.set_title("Throughput (mean over folds)")
    ax_fps.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    speed_png = RESULTS_DIR / f"{PROJECT_NAME}_inference_speed.png"
    fig_spd.savefig(speed_png, bbox_inches="tight", dpi=150)
    plt.show()
    print("Saved:", speed_png)
else:
    print(f"Inference run CSV not found; skipping speed plot. Expected: {infer_run_csv}")

# --- 5. Combined training-vs-inference plots ---
infer_model_csv = RESULTS_DIR / f"{PROJECT_NAME}_model_metrics_inference.csv"
combined_bar_path     = RESULTS_DIR / f"{PROJECT_NAME}_bar_plots_combined.png"
combined_scatter_path = RESULTS_DIR / f"{PROJECT_NAME}_scatter_plots_combined.png"

if infer_run_csv.exists() and infer_model_csv.exists():
    inference_runs_df  = pd.read_csv(infer_run_csv)
    inference_model_df = pd.read_csv(infer_model_csv)

    train_metric_shorts = [c[len("train_"):].rsplit("__", 1)[0]
                           for c in model_summary_display.columns
                           if c.endswith("__mean") and c.startswith("train_")]
    infer_metric_shorts = [c[len("infer_"):].rsplit("__", 1)[0]
                           for c in inference_model_df.columns
                           if c.endswith("__mean") and c.startswith("infer_")]

    common_metrics = [m for m in train_metric_shorts if m in infer_metric_shorts]
    if common_metrics:
        n_metrics = len(common_metrics)
        fig_cb, axes_cb = plt.subplots(1, n_metrics, figsize=(5 * n_metrics, 5), squeeze=False)
        axes_cb = axes_cb[0]
        combined_models = list(model_summary_display["model_base"].unique())
        x     = np.arange(len(combined_models))
        width = 0.35

        for ax, metric_short in zip(axes_cb, common_metrics):
            tm = f"train_{metric_short}__mean"; ts = f"train_{metric_short}__std"
            im = f"infer_{metric_short}__mean"; is_ = f"infer_{metric_short}__std"

            t_means, t_stds, i_means, i_stds = [], [], [], []
            for m in combined_models:
                tr = model_summary_display[model_summary_display["model_base"] == m]
                ir = inference_model_df[inference_model_df["model_base"] == m]
                t_means.append(float(tr.iloc[0][tm]) if (not tr.empty and tm in tr.columns) else np.nan)
                t_stds.append(float(tr.iloc[0].get(ts, 0.0)) if not tr.empty else 0.0)
                i_means.append(float(ir.iloc[0][im]) if (not ir.empty and im in ir.columns) else np.nan)
                i_stds.append(float(ir.iloc[0].get(is_, 0.0)) if not ir.empty else 0.0)

            for i, m in enumerate(combined_models):
                ax.bar(x[i] - width/2, t_means[i], width=width,
                       yerr=t_stds[i] if not pd.isna(t_means[i]) else None,
                       color=model_colors.get(m, (0.5, 0.5, 0.5)),
                       capsize=4, label="train" if i == 0 else None)
                ax.bar(x[i] + width/2, i_means[i], width=width,
                       yerr=i_stds[i] if not pd.isna(i_means[i]) else None,
                       color=model_colors.get(m, (0.5, 0.5, 0.5)),
                       hatch="////", edgecolor="black", alpha=0.9,
                       capsize=4, label="inference" if i == 0 else None)
            ax.set_title(f"{metric_short} (train vs inference)"); ax.set_ylim(0, 1)
            ax.set_xticks(x); ax.set_xticklabels(combined_models, rotation=45)
            if metric_short == common_metrics[0]:
                ax.legend()

        plt.tight_layout()
        fig_cb.savefig(combined_bar_path, bbox_inches="tight", dpi=150)
        plt.show()
        print("Saved:", combined_bar_path)
    else:
        print("No common metrics found between training and inference; skipping combined plots.")
else:
    print("Inference CSVs not present; skipping combined plots.")

In [ ]:
# === Cell 11: Evaluation Graph Widget ===

# --- Plot style (10 pt at 16 cm LaTeX text width) ---
_FW        = 16 / 2.54
_FONT_SIZE = 10

plt.rcParams["font.family"]     = "DejaVu Sans"
plt.rcParams["font.size"]       = _FONT_SIZE
plt.rcParams["axes.titlesize"]  = _FONT_SIZE + 1
plt.rcParams["axes.labelsize"]  = _FONT_SIZE
plt.rcParams["legend.fontsize"] = _FONT_SIZE - 1
plt.rcParams["savefig.dpi"]     = 150

# --- Fixed colour per model ---
MODEL_COLORS = {
    "yolov8n-seg": "#1f77b4",
    "yolov8s-seg": "#ff7f0e",
    "yolov8m-seg": "#2ca02c",
    "yolo11n-seg": "#d62728",
    "yolo11s-seg": "#9467bd",
    "yolo11m-seg": "#8c564b",
    "yolo26n-seg": "#e377c2",
    "yolo26s-seg": "#7f7f7f",
    "yolo26m-seg": "#bcbd22",
}

def _model_color(model_base: str):
    """Return the model colour; grey if unknown."""
    return MODEL_COLORS.get(model_base, (0.5, 0.5, 0.5))

# --- Load saved metric CSVs (cell can run independently of Cells 9-10) ---
training_model_path   = RESULTS_DIR / f"{PROJECT_NAME}_model_metrics_training.csv"
inference_model_path  = RESULTS_DIR / f"{PROJECT_NAME}_model_metrics_inference.csv"
training_run_path     = RESULTS_DIR / f"{PROJECT_NAME}_run_metrics_training.csv"
inference_run_path    = RESULTS_DIR / f"{PROJECT_NAME}_run_metrics_inference.csv"

train_model_df = pd.read_csv(training_model_path)  if training_model_path.exists()  else None
infer_model_df = pd.read_csv(inference_model_path) if inference_model_path.exists() else None
train_run_df   = pd.read_csv(training_run_path)    if training_run_path.exists()    else None
infer_run_df   = pd.read_csv(inference_run_path)   if inference_run_path.exists()   else None

# --- Metric discovery helpers ---
PREFIXES = ["train_", "infer_", "test_", "val_", "metrics/", "metric/"]

def strip_prefixes(name: str) -> str:
    """Repeatedly strip any of PREFIXES from the start of name."""
    s = name
    changed = True
    while changed:
        changed = False
        for p in PREFIXES:
            if s.startswith(p):
                s = s[len(p):]
                changed = True
    return s

def discover_metrics(train_df, infer_df):
    """Build {metric_short: {'train': {...}, 'infer': {...}}} from CSV column names."""
    metrics = {}
    def add_side(df, side):
        if df is None:
            return
        for col in df.columns:
            if not col.endswith("__mean"):
                continue
            base = col.rsplit("__", 1)[0]
            metric_short = strip_prefixes(base)
            entry = metrics.setdefault(metric_short, {
                "train": {"mean_col": None, "std_col": None},
                "infer": {"mean_col": None, "std_col": None}
            })
            entry[side]["mean_col"] = col
            std_col = f"{base}__std"
            if std_col in df.columns:
                entry[side]["std_col"] = std_col
    add_side(train_df, "train")
    add_side(infer_df, "infer")
    return metrics

metrics = discover_metrics(train_model_df, infer_model_df)

def find_run_column(run_df, metric_short: str):
    """Find first run-level column matching metric_short (after prefix strip)."""
    if run_df is None:
        return None
    exact, fallback = None, []
    for col in run_df.columns:
        clean = strip_prefixes(col)
        if clean == metric_short:
            exact = col
            break
        if clean.endswith(metric_short) or metric_short in clean:
            fallback.append(col)
    return exact or (fallback[0] if fallback else None)

# --- Models ---
all_models = set()
for df in (train_model_df, infer_model_df):
    if df is not None and "model_base" in df.columns:
        all_models.update(df["model_base"].unique())
all_models = sorted(all_models)

# model_colors is also used by the Statistical Analysis cell.
model_colors = {m: _model_color(m) for m in all_models}

def size_group_order_key(size_val):
    """Sort key putting (m, s, n, l, x) in that order."""
    if pd.isna(size_val):
        return (10, "zz")
    s = str(size_val).lower()
    pri_map = {"m": 0, "s": 1, "n": 2, "l": 3, "x": 4}
    return (pri_map.get(s[:1], 5), s)

# --- Widgets ---
metric_options = sorted(metrics.keys())
if not metric_options:
    print("No aggregated metric columns with '__mean' found in training or inference CSVs.")

metric_dropdown = widgets.Dropdown(
    options=metric_options or ["<no metrics>"],
    value=metric_options[0] if metric_options else None,
    description="Metric:",
    disabled=not metric_options,
    layout=widgets.Layout(width="300px"),
)
include_train_checkbox = widgets.Checkbox(value=True, description="Include training (val)")
include_infer_checkbox = widgets.Checkbox(value=True, description="Include inference (test)")
grouping_dropdown = widgets.Dropdown(
    options=["none", "model_size", "model_version"],
    value="none",
    description="Group by:",
)
combine_bars_checkbox = widgets.Checkbox(value=False, description="Combine bars")
refresh_button = widgets.Button(description="Refresh plot",
                                button_style="primary",
                                layout=widgets.Layout(width="150px"))
save_button    = widgets.Button(description="Save plots to RESULTS_DIR",
                                button_style="",
                                layout=widgets.Layout(width="220px"))
graph_output = widgets.Output()
last_figs    = {}   # {"bar": fig, "scatter": fig}

display(
    widgets.VBox([
        metric_dropdown,
        widgets.HBox([include_train_checkbox, include_infer_checkbox]),
        widgets.HBox([grouping_dropdown, combine_bars_checkbox]),
        widgets.HBox([refresh_button, save_button]),
    ]),
    graph_output,
)

# --- Shared data assembly (used by both bar and scatter renderers) ---
def _build_plot_data(metric_short, include_train, include_infer, grouping):
    """Assemble ordered rows, x positions, axis limits for the selected metric."""
    info       = metrics[metric_short]
    train_info = info["train"]
    infer_info = info["infer"]

    if include_train and (train_model_df is None or train_info["mean_col"] is None):
        include_train = False
    if include_infer and (infer_model_df is None or infer_info["mean_col"] is None):
        include_infer = False
    if not include_train and not include_infer:
        return None

    rows = []
    for m_base in all_models:
        r = {"model_base": m_base,
             "train_mean": np.nan, "train_std": np.nan,
             "infer_mean": np.nan, "infer_std": np.nan,
             "model_size": None, "model_version": None}
        for df in (train_model_df, infer_model_df):
            if df is None or "model_base" not in df.columns:
                continue
            sub = df[df["model_base"] == m_base]
            if sub.empty:
                continue
            if r["model_size"] is None and "model_size" in sub.columns:
                r["model_size"] = sub.iloc[0].get("model_size")
            if r["model_version"] is None and "model_version" in sub.columns:
                r["model_version"] = sub.iloc[0].get("model_version")
        if include_train and train_model_df is not None and train_info["mean_col"] in train_model_df.columns:
            sub = train_model_df[train_model_df["model_base"] == m_base]
            if not sub.empty:
                r["train_mean"] = float(sub.iloc[0][train_info["mean_col"]])
                if train_info["std_col"]:
                    r["train_std"] = float(sub.iloc[0].get(train_info["std_col"], np.nan))
        if include_infer and infer_model_df is not None and infer_info["mean_col"] in infer_model_df.columns:
            sub = infer_model_df[infer_model_df["model_base"] == m_base]
            if not sub.empty:
                r["infer_mean"] = float(sub.iloc[0][infer_info["mean_col"]])
                if infer_info["std_col"]:
                    r["infer_std"] = float(sub.iloc[0].get(infer_info["std_col"], np.nan))
        rows.append(r)

    if not rows:
        return None

    metric_lower   = metric_short.lower()
    is_time_metric = any(tok in metric_lower for tok in ["time", "latency", "ms", "sec", "s/", "speed"])
    ascending      = True if is_time_metric else False

    for r in rows:
        vals = []
        if include_train and not np.isnan(r["train_mean"]): vals.append(r["train_mean"])
        if include_infer and not np.isnan(r["infer_mean"]): vals.append(r["infer_mean"])
        r["sort_score"] = float(np.mean(vals)) if vals else np.nan

    def sort_key(r):
        s = r["sort_score"]
        if np.isnan(s):
            return (float("inf") if ascending else float("-inf"), r["model_base"])
        v = s if ascending else -s
        return (v, r["model_base"])

    groups = {}
    if grouping == "none":
        groups[None] = rows
    else:
        for r in rows:
            g = r["model_size"] if grouping == "model_size" else r["model_version"]
            groups.setdefault(g, []).append(r)

    if grouping == "model_size":
        group_keys = sorted(groups.keys(), key=size_group_order_key)
    elif grouping == "model_version":
        group_keys = sorted(groups.keys(), key=lambda x: "" if x is None else str(x))
    else:
        group_keys = [None]

    x_positions  = {}
    x_val        = 0.0
    group_gap    = 1.0
    ordered_rows = []
    for gi, gk in enumerate(group_keys):
        members = sorted(groups[gk], key=sort_key)
        ordered_rows.extend(members)
        for r in members:
            x_positions[r["model_base"]] = x_val
            x_val += 1.0
        if gi != len(group_keys) - 1:
            x_val += group_gap

    return {
        "ordered_rows":  ordered_rows,
        "x_positions":   x_positions,
        "include_train": include_train,
        "include_infer": include_infer,
        "is_time_metric": is_time_metric,
        "train_info":    train_info,
        "infer_info":    infer_info,
    }


def _axis_limits(all_vals, is_time_metric):
    """Compute shared y-axis limits."""
    if not all_vals:
        return 0, 1
    ymax = max(all_vals); ymin = min(all_vals)
    if is_time_metric:
        return 0.0, (ymax * 1.1 if ymax > 0 else 1)
    lower = 0.0 if ymin >= 0 else ymin * 1.1
    upper = 1.0 if ymax <= 1.0 else ymax * 1.1
    return lower, upper


def _add_bands(ax, x_ticks):
    """Alternating light-grey / white vertical bands."""
    for i, xt in enumerate(x_ticks):
        if i % 2 == 0:
            ax.axvspan(xt - 0.5, xt + 0.5, color="#f0f0f0", zorder=0)


# --- Rendering ---
def update_plot(_=None):
    """Render the interactive comparison plots for the currently selected metric."""
    global last_figs
    with graph_output:
        clear_output(wait=True)

        if train_model_df is None and infer_model_df is None:
            print("No aggregated CSVs found in RESULTS_DIR. Nothing to plot.")
            return

        metric_short = metric_dropdown.value
        if not metric_short or metric_short not in metrics:
            print("No valid metric selected.")
            return

        include_train = include_train_checkbox.value
        include_infer = include_infer_checkbox.value
        if not include_train and not include_infer:
            print("Please select at least one side (training or inference).")
            return

        grouping = grouping_dropdown.value
        pd_data  = _build_plot_data(metric_short, include_train, include_infer, grouping)
        if pd_data is None:
            print("No data available for the selected metric.")
            return

        ordered_rows = pd_data["ordered_rows"]
        x_positions  = pd_data["x_positions"]
        include_train = pd_data["include_train"]
        include_infer = pd_data["include_infer"]
        is_time_metric = pd_data["is_time_metric"]

        train_run_col = find_run_column(train_run_df, metric_short) if train_run_df is not None else None
        infer_run_col = find_run_column(infer_run_df, metric_short) if infer_run_df is not None else None

        combine_bars = combine_bars_checkbox.value and include_train and include_infer
        bar_width    = 0.35 if not combine_bars else 0.6
        offset       = 0.2 if not combine_bars else 0.0

        x_ticks = [x_positions[r["model_base"]] for r in ordered_rows]
        labels  = [r["model_base"] for r in ordered_rows]

        # --- Figure 1: bar chart (mean ± std) ---
        plt.close("all")
        fig_bar, ax_bar = plt.subplots(figsize=(_FW, _FW * 0.55))
        _add_bands(ax_bar, x_ticks)

        all_vals_bar = []
        for r in ordered_rows:
            m     = r["model_base"]
            x0    = x_positions[m]
            color = _model_color(m)

            if combine_bars:
                x_train = x_infer = x0
            else:
                if include_train and include_infer:
                    x_train, x_infer = x0 - offset, x0 + offset
                else:
                    x_train = x_infer = x0

            if combine_bars:
                vals = []
                if include_train and not np.isnan(r["train_mean"]): vals.append(r["train_mean"])
                if include_infer and not np.isnan(r["infer_mean"]): vals.append(r["infer_mean"])
                if vals:
                    mean_c = float(np.mean(vals))
                    stds   = []
                    if include_train and not np.isnan(r["train_std"]): stds.append(r["train_std"] ** 2)
                    if include_infer and not np.isnan(r["infer_std"]): stds.append(r["infer_std"] ** 2)
                    std_c  = float(np.sqrt(np.mean(stds))) if stds else None
                    ax_bar.bar(x0, mean_c, width=bar_width,
                               yerr=std_c if std_c is not None else None,
                               color=color, edgecolor="black", hatch="..", capsize=4)
                    all_vals_bar.append(mean_c)
            else:
                if include_train and not np.isnan(r["train_mean"]):
                    ax_bar.bar(x_train, r["train_mean"], width=bar_width,
                               yerr=r["train_std"] if not np.isnan(r["train_std"]) else None,
                               color=color, edgecolor="black", alpha=0.9, capsize=4)
                    all_vals_bar.append(r["train_mean"])
                if include_infer and not np.isnan(r["infer_mean"]):
                    ax_bar.bar(x_infer, r["infer_mean"], width=bar_width,
                               yerr=r["infer_std"] if not np.isnan(r["infer_std"]) else None,
                               color=color, hatch="////", edgecolor="black", alpha=0.9, capsize=4)
                    all_vals_bar.append(r["infer_mean"])

        ax_bar.set_xticks(x_ticks)
        ax_bar.set_xticklabels(labels, rotation=45, ha="right")
        ax_bar.set_ylabel("Aggregated Value")
        title_metric = metric_short.replace("_", " ").title()
        ax_bar.set_title(f"{title_metric} — Aggregated")
        ax_bar.grid(axis="y", alpha=0.3)

        lower, upper = _axis_limits(all_vals_bar, is_time_metric)
        ax_bar.set_ylim(lower, upper)

        bar_legend_handles = []
        if combine_bars:
            bar_legend_handles.append(Patch(facecolor="lightgray", edgecolor="black",
                                            hatch="..", label="Combined"))
        else:
            if include_train:
                bar_legend_handles.append(Patch(facecolor="white", edgecolor="black",
                                                label="Train (Val)"))
            if include_infer:
                bar_legend_handles.append(Patch(facecolor="white", edgecolor="black",
                                                hatch="////", label="Inference (Test)"))
        if bar_legend_handles:
            ax_bar.legend(handles=bar_legend_handles, loc="best",
                          fontsize=_FONT_SIZE - 1)

        fig_bar.tight_layout()
        display(fig_bar)

        # --- Figure 2: scatter (per-run values and mean lines) ---
        fig_scat, ax_scat = plt.subplots(figsize=(_FW, _FW * 0.55))
        _add_bands(ax_scat, x_ticks)

        all_vals_scat = []
        for r in ordered_rows:
            m     = r["model_base"]
            x0    = x_positions[m]
            color = _model_color(m)

            if combine_bars:
                x_train = x_infer = x0
            else:
                if include_train and include_infer:
                    x_train, x_infer = x0 - offset, x0 + offset
                else:
                    x_train = x_infer = x0

            if include_train and train_run_df is not None and train_run_col is not None and train_run_col in train_run_df.columns:
                vals_t = train_run_df.loc[train_run_df["model_base"] == m, train_run_col].dropna().values
                if len(vals_t) > 0:
                    xs = np.full(len(vals_t), x_train, float)
                    jitter = np.random.uniform(-0.05, 0.05, size=len(vals_t))
                    ax_scat.scatter(xs + jitter, vals_t, marker="o", s=35,
                                   alpha=0.8, color=color, edgecolor="k")
                    all_vals_scat.extend(vals_t)
            if include_infer and infer_run_df is not None and infer_run_col is not None and infer_run_col in infer_run_df.columns:
                vals_i = infer_run_df.loc[infer_run_df["model_base"] == m, infer_run_col].dropna().values
                if len(vals_i) > 0:
                    xs = np.full(len(vals_i), x_infer, float)
                    jitter = np.random.uniform(-0.05, 0.05, size=len(vals_i))
                    ax_scat.scatter(xs + jitter, vals_i, marker="D", s=40,
                                   alpha=0.8, color=color, edgecolor="k")
                    all_vals_scat.extend(vals_i)

            line_span = 0.2
            if include_train and not np.isnan(r["train_mean"]):
                ax_scat.hlines(r["train_mean"], x_train - line_span, x_train + line_span,
                               color="black", linewidth=1, linestyle="-")
                all_vals_scat.append(r["train_mean"])
            if include_infer and not np.isnan(r["infer_mean"]):
                ax_scat.hlines(r["infer_mean"], x_infer - line_span, x_infer + line_span,
                               color="black", linewidth=1, linestyle="--")
                all_vals_scat.append(r["infer_mean"])

        ax_scat.set_xticks(x_ticks)
        ax_scat.set_xticklabels(labels, rotation=45, ha="right")
        ax_scat.set_ylabel("Per-Run Value")
        ax_scat.set_xlabel("Model")
        ax_scat.set_title(f"{title_metric} — Per-Run Values")
        ax_scat.grid(axis="y", alpha=0.3)

        lower_s, upper_s = _axis_limits(all_vals_scat, is_time_metric)
        ax_scat.set_ylim(lower_s, upper_s)

        scatter_legend_handles = []
        if include_train and train_run_df is not None and train_run_col is not None and train_run_col in train_run_df.columns:
            scatter_legend_handles.append(Line2D([], [], marker="o", linestyle="None",
                                                  color="black", label="Train Runs"))
        if include_infer and infer_run_df is not None and infer_run_col is not None and infer_run_col in infer_run_df.columns:
            scatter_legend_handles.append(Line2D([], [], marker="D", linestyle="None",
                                                  color="black", label="Inference Runs"))
        if include_train:
            scatter_legend_handles.append(Line2D([], [], linestyle="-", color="black",
                                                 linewidth=3, label="Train Mean"))
        if include_infer:
            scatter_legend_handles.append(Line2D([], [], linestyle="--", color="black",
                                                 linewidth=3, label="Inference Mean"))
        if scatter_legend_handles:
            ax_scat.legend(handles=scatter_legend_handles, loc="best",
                           fontsize=_FONT_SIZE - 1)

        fig_scat.tight_layout()
        display(fig_scat)

        last_figs = {"bar": fig_bar, "scatter": fig_scat}


def save_plot(_=None):
    """Persist both figures as separate PNGs under RESULTS_DIR."""
    if not last_figs:
        print("No plots to save. Please click 'Refresh plot' first.")
        return
    metric_short = metric_dropdown.value or "metric"
    safe_metric  = re.sub(r"[^a-zA-Z0-9_]+", "_", metric_short)
    for key, fig in last_figs.items():
        out_path = RESULTS_DIR / f"{PROJECT_NAME}_interactive_{safe_metric}_{key}.png"
        fig.savefig(out_path, bbox_inches="tight", dpi=150)
        print(f"Saved {key} plot: {out_path}")

refresh_button.on_click(update_plot)
save_button.on_click(save_plot)

In [ ]:
# === Cell 12: Statistical Analysis ===

# --- Plot style (10 pt at 16 cm LaTeX text width) ---
_FW        = 16 / 2.54
_FONT_SIZE = 10

plt.rcParams["font.family"]     = "DejaVu Sans"
plt.rcParams["font.size"]       = _FONT_SIZE
plt.rcParams["axes.titlesize"]  = _FONT_SIZE + 1
plt.rcParams["axes.labelsize"]  = _FONT_SIZE
plt.rcParams["legend.fontsize"] = _FONT_SIZE - 1
plt.rcParams["savefig.dpi"]     = 150

# --- Method ---
# Friedman test (folds as blocks), pairwise Wilcoxon signed-rank tests with Holm
# correction, and 95% percentile-bootstrap CIs (n_boot = 5000) of the mean paired
# difference. n = NUM_FOLDS correlated observations per model; tests have low power.

from scipy import stats

# --- Preconditions ---
required_globals = ["metric_dropdown", "train_run_df", "infer_run_df",
                    "find_run_column", "strip_prefixes"]
missing = [name for name in required_globals if name not in globals()]
if missing:
    raise RuntimeError(
        "The following globals are missing:\n  " + ", ".join(missing)
        + "\n\nPlease run Cell 11 (Evaluation Graph Widget) first."
    )

if "parse_model_version_size" not in globals():
    def parse_model_version_size(model_name: str):
        """Fallback parser if the Training cell has not run in this session."""
        stem = Path(model_name).stem
        base = stem.split("-")[0]
        if not base.startswith("yolo") or len(base) < 6:
            return base, None
        return base[:-1], base[-1]

# --- Widgets ---
data_source_dropdown = widgets.Dropdown(
    options=[
        ("Training runs (validation)", "train"),
        ("Inference runs (test)",      "infer"),
        ("Both training + inference",  "both"),
    ],
    value="infer",
    description="Data:",
    layout=widgets.Layout(width="280px"),
)
stat_grouping_dropdown = widgets.Dropdown(
    options=[
        ("No grouping (all models)", "none"),
        ("By model_size",            "model_size"),
        ("By model_version",         "model_version"),
    ],
    value="none",
    description="Group by:",
    layout=widgets.Layout(width="280px"),
)
alpha_float = widgets.BoundedFloatText(
    value=0.05, min=0.0001, max=0.5, step=0.005,
    description="alpha:", layout=widgets.Layout(width="150px"),
)
run_button = widgets.Button(
    description="Run statistical analysis",
    button_style="primary", layout=widgets.Layout(width="250px"),
)
stat_output = widgets.Output()

display(
    widgets.VBox([
        widgets.HTML("<b>Cell 12: Friedman + Wilcoxon comparison of models</b>"),
        widgets.HBox([widgets.Label("Metric (from Cell 11):"), metric_dropdown]),
        widgets.HBox([data_source_dropdown, stat_grouping_dropdown, alpha_float]),
        run_button,
        stat_output,
    ])
)

# --- Core helpers ---
def _ensure_model_metadata(df: pd.DataFrame) -> pd.DataFrame:
    """Add model_version / model_size columns if absent."""
    df = df.copy()
    if ("model_version" not in df.columns) or ("model_size" not in df.columns):
        versions, sizes = zip(*df["model_base"].apply(parse_model_version_size))
        df["model_version"] = versions
        df["model_size"]    = sizes
    return df

def _build_run_level_df(metric_short: str, source: str) -> pd.DataFrame:
    """Stack train/infer run-level rows for the chosen metric into one long-form df."""
    frames = []
    if source in ("train", "both") and train_run_df is not None:
        train_col = find_run_column(train_run_df, metric_short)
        if train_col is not None and train_col in train_run_df.columns:
            df_t = train_run_df.copy()
            cols = ["model_base", train_col]
            for c in ("model_version", "model_size", "fold_idx"):
                if c in df_t.columns:
                    cols.append(c)
            df_t = df_t[cols].rename(columns={train_col: "value"})
            df_t["source"] = "train"
            frames.append(df_t.dropna(subset=["value"]))
    if source in ("infer", "both") and infer_run_df is not None:
        infer_col = find_run_column(infer_run_df, metric_short)
        if infer_col is not None and infer_col in infer_run_df.columns:
            df_i = infer_run_df.copy()
            cols = ["model_base", infer_col]
            for c in ("model_version", "model_size", "fold_idx"):
                if c in df_i.columns:
                    cols.append(c)
            df_i = df_i[cols].rename(columns={infer_col: "value"})
            df_i["source"] = "infer"
            frames.append(df_i.dropna(subset=["value"]))
    if not frames:
        return pd.DataFrame(columns=["model_base", "model_version", "model_size",
                                     "fold_idx", "value", "source"])
    df = pd.concat(frames, ignore_index=True)
    df = _ensure_model_metadata(df)
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df.dropna(subset=["value"])

def _is_time_metric(metric_short: str) -> bool:
    """Return True if the metric is a latency-like measure (lower is better)."""
    metric_lower = str(metric_short).lower()
    return any(tok in metric_lower for tok in ["time", "latency", "ms", "sec", "s/", "speed"])

def _build_block_matrix(df: pd.DataFrame):
    """
    Reshape long df into a (n_folds, n_models) matrix for Friedman / Wilcoxon.

    Returns (matrix, fold_index, models), or (None, None, None) if any cell is missing.
    """
    if "fold_idx" not in df.columns:
        return None, None, None
    wide = (df.pivot_table(index=["source", "fold_idx"],
                           columns="model_base",
                           values="value",
                           aggfunc="mean")
              .sort_index())
    if wide.isna().any().any() or wide.shape[0] < 2 or wide.shape[1] < 2:
        return None, None, None
    return wide.values, list(wide.index), list(wide.columns)

def _friedman_test(matrix: np.ndarray):
    """Run Friedman chi-square test on a (n_blocks, n_groups) matrix."""
    if matrix is None or matrix.shape[1] < 2:
        return None
    args = [matrix[:, j] for j in range(matrix.shape[1])]
    try:
        stat, p = stats.friedmanchisquare(*args)
    except ValueError as e:
        return {"error": str(e)}
    n_blocks, n_groups = matrix.shape
    # Kendall's W effect size: W = chi2 / (n_blocks * (n_groups - 1))
    W = float(stat) / (n_blocks * (n_groups - 1)) if (n_blocks > 0 and n_groups > 1) else np.nan
    return {
        "stat":     float(stat),
        "p":        float(p),
        "n_blocks": int(n_blocks),
        "n_groups": int(n_groups),
        "kendall_w": float(W),
    }

def _bootstrap_paired_ci(diff_values: np.ndarray, alpha: float = 0.05, n_boot: int = 5000, seed: int = 0):
    """Percentile bootstrap CI for the mean of a vector of paired differences."""
    if len(diff_values) == 0:
        return np.nan, np.nan
    rng       = np.random.default_rng(seed)
    n         = len(diff_values)
    idx       = rng.integers(0, n, size=(n_boot, n))
    boot_means = diff_values[idx].mean(axis=1)
    low  = float(np.quantile(boot_means, alpha / 2.0))
    high = float(np.quantile(boot_means, 1 - alpha / 2.0))
    return low, high

def _wilcoxon_pairwise(matrix: np.ndarray, models, alpha: float):
    """
    Pairwise Wilcoxon signed-rank tests on per-fold paired differences,
    with Holm-Bonferroni correction across all (k choose 2) pairs.
    """
    rows = []
    for i, m_a in enumerate(models):
        for j in range(i + 1, len(models)):
            m_b = models[j]
            a, b  = matrix[:, i], matrix[:, j]
            diff = a - b
            if np.all(diff == 0):
                p_raw = 1.0
            else:
                try:
                    _, p_raw = stats.wilcoxon(a, b, zero_method="wilcox", correction=False,
                                              alternative="two-sided", mode="auto")
                except ValueError:
                    p_raw = np.nan
            mean_diff       = float(np.mean(diff))
            ci_low, ci_high = _bootstrap_paired_ci(diff, alpha=alpha, n_boot=5000)
            rows.append({
                "model_A":    m_a,
                "model_B":    m_b,
                "mean_diff":  mean_diff,
                "ci_low":     ci_low,
                "ci_high":    ci_high,
                "p_value_raw": float(p_raw) if not np.isnan(p_raw) else np.nan,
            })
    if not rows:
        return pd.DataFrame()
    df_p = pd.DataFrame(rows).sort_values("p_value_raw", na_position="last").reset_index(drop=True)
    # Holm-Bonferroni adjustment
    m_total = df_p["p_value_raw"].notna().sum()
    p_adj   = []
    rank_i  = 0
    for _, row in df_p.iterrows():
        p_raw = row["p_value_raw"]
        if np.isnan(p_raw):
            p_adj.append(np.nan)
            continue
        rank_i += 1
        p_adj.append(min(1.0, p_raw * (m_total - rank_i + 1)))
    df_p["p_value_adj"] = p_adj
    df_p["significant"] = df_p["p_value_adj"].apply(
        lambda v: (False if (v is None or (isinstance(v, float) and np.isnan(v))) else v < alpha)
    )
    return df_p[["model_A", "model_B", "mean_diff", "ci_low", "ci_high", "p_value_adj", "significant"]]


def _plot_ci_pairs(df_pairs: pd.DataFrame):
    """Horizontal CI plot of pairwise mean differences (95% bootstrap CIs)."""
    if df_pairs is None or df_pairs.empty:
        print("No pairwise comparisons to plot.")
        return

    rows   = df_pairs.reset_index(drop=True)
    n      = len(rows)
    y_pos  = np.arange(n)
    lowers = rows["ci_low"].values
    uppers = rows["ci_high"].values
    means  = rows["mean_diff"].values

    # Model colours from the Evaluation Graph Widget cell; grey if unavailable
    _mc        = globals().get("model_colors", {})
    row_colors = [_mc.get(a, "#555555") for a in rows["model_A"]]

    fig, ax = plt.subplots(figsize=(_FW, max(5 / 2.54, n * 0.38)))

    for i, (lo, hi, mn, c) in enumerate(zip(lowers, uppers, means, row_colors)):
        ax.hlines(i, lo, hi, color="black", linewidth=1.4, zorder=2)
        ax.plot(lo, i, "o", color="black", markersize=5, zorder=3)
        ax.plot(hi, i, "o", color="black", markersize=5, zorder=3)
        ax.plot(mn, i, "x", color="black", markersize=7,
                markeredgewidth=1.8, zorder=4)

    ax.vlines(0, -0.5, n - 0.5, linestyles="dashed",
              color="grey", linewidth=0.9, zorder=1)

    labels = [f"{a} − {b}" for a, b in zip(rows["model_A"], rows["model_B"])]
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=_FONT_SIZE - 1)
    ax.set_xlabel("Mean Difference (95% Bootstrap CI)")
    ax.set_title("Pairwise Comparisons")
    ax.grid(axis="x", linestyle=":", alpha=0.5)
    fig.tight_layout()
    plt.show()
    return fig


def _build_text_summary(summary_df, pairwise_df, metric_short, data_label,
                         group_label, alpha, higher_is_better, friedman_p):
    """Build a text summary of the best model and significant pairwise differences."""
    if summary_df.empty:
        return "No data available for summary."
    sort_asc       = not higher_is_better
    summary_sorted = summary_df.sort_values("mean", ascending=sort_asc)
    best_model     = summary_sorted.iloc[0]["model_base"]
    p_str          = f"{friedman_p:.4g}" if friedman_p is not None and not np.isnan(friedman_p) else "n/a"

    if pairwise_df is None or pairwise_df.empty:
        return (f"For metric {metric_short} ({data_label}) in {group_label}, "
                f"model {best_model} has the best mean. Friedman p = {p_str}.")

    better_than, not_better = [], []
    for other in summary_df["model_base"].unique():
        if other == best_model:
            continue
        mask1 = (pairwise_df["model_A"] == best_model) & (pairwise_df["model_B"] == other)
        mask2 = (pairwise_df["model_A"] == other)      & (pairwise_df["model_B"] == best_model)
        if mask1.any():
            row = pairwise_df.loc[mask1].iloc[0]; diff = float(row["mean_diff"])
        elif mask2.any():
            row = pairwise_df.loc[mask2].iloc[0]; diff = -float(row["mean_diff"])
        else:
            not_better.append(other); continue
        if not row["significant"]:
            not_better.append(other); continue
        if higher_is_better:
            (better_than if diff > 0 else not_better).append(other)
        else:
            (better_than if diff < 0 else not_better).append(other)

    group_text = "all models" if group_label is None else f"group '{group_label}'"
    if better_than:
        better_list = ", ".join(sorted(better_than))
        if not_better:
            not_list = ", ".join(sorted(not_better))
            return (f"For metric {metric_short} ({data_label}) in {group_text}, "
                    f"model {best_model} has the best mean and is significantly better "
                    f"than {better_list} (alpha = {alpha:.3f}), but not significantly "
                    f"different from {not_list}. Friedman p = {p_str}.")
        return (f"For metric {metric_short} ({data_label}) in {group_text}, "
                f"model {best_model} has the best mean and is significantly better "
                f"than all other models considered (alpha = {alpha:.3f}). "
                f"Friedman p = {p_str}.")
    return (f"For metric {metric_short} ({data_label}) in {group_text}, "
            f"model {best_model} has the best mean, but no statistically significant "
            f"pairwise differences were found at alpha = {alpha:.3f}. "
            f"Friedman p = {p_str}.")

def _analyze_one_subset(df: pd.DataFrame, metric_short, data_label, group_label, alpha):
    """Run summary + Friedman + Wilcoxon + bootstrap CIs for one data subset."""
    if df.empty:
        print("No data for this subset.")
        return

    higher_is_better = not _is_time_metric(metric_short)

    summary_df = (
        df.groupby("model_base")
          .agg(mean   =("value", "mean"),
               std    =("value", "std"),
               median =("value", "median"),
               iqr    =("value", lambda x: float(np.subtract(*np.percentile(x, [75, 25]))) if len(x) else np.nan),
               n_runs =("value", "count"))
          .reset_index()
          .sort_values("mean", ascending=not higher_is_better)
    )

    print("=" * 80)
    header = f"Metric: {metric_short} | Data: {data_label}"
    if group_label is not None:
        header += f" | Group: {group_label}"
    print(header)
    print("-" * 80)
    print("Per-model summary:")
    display(summary_df[["model_base", "n_runs", "mean", "std", "median", "iqr"]].round(6))

    if summary_df["n_runs"].sum() < 3 or summary_df["model_base"].nunique() < 2:
        print("Not enough data for statistical comparison (need >= 2 models and >= 3 total runs).")
        return

    matrix, fold_index, models = _build_block_matrix(df)
    if matrix is None:
        print("Block matrix could not be built (unbalanced design or missing folds).")
        print("Friedman / Wilcoxon require a complete fold-by-model matrix — falling back to "
              "per-model summary only.")
        return

    print(f"\nBlock matrix shape: n_blocks={matrix.shape[0]}, n_models={matrix.shape[1]}")
    print(f"Models in order: {models}")

    fried = _friedman_test(matrix)
    if fried is None or "error" in (fried or {}):
        err_msg = (fried or {}).get("error", "unknown")
        print(f"Friedman test could not be computed: {err_msg}")
        return

    print("\nFriedman test results:")
    display(pd.DataFrame([{
        "chi2":             fried["stat"],
        "df":               fried["n_groups"] - 1,
        "p_value":          fried["p"],
        "kendall_w":        fried["kendall_w"],
        "n_blocks (folds)": fried["n_blocks"],
        "n_groups (models)":fried["n_groups"],
    }]).round(6))

    p = fried["p"]
    if p < alpha:
        print(f"Friedman p = {p:.4g} < alpha = {alpha:.3f} -> at least one model differs.")
        do_posthoc = True
    else:
        print(f"Friedman p = {p:.4g} >= alpha = {alpha:.3f} -> no overall difference detected "
              "(at n=4 this is likely a power-limited result; see caveats at top of cell).")
        do_posthoc = False

    pairwise_df = pd.DataFrame()
    if do_posthoc:
        print("\nPairwise Wilcoxon signed-rank (paired per-fold), Holm-Bonferroni adjusted:")
        pairwise_df = _wilcoxon_pairwise(matrix, models, alpha=alpha)
        if pairwise_df.empty:
            print("No pairwise post-hoc results.")
        else:
            display(pairwise_df.round(6))
            sig_pairs = pairwise_df.loc[pairwise_df["significant"]]
            if sig_pairs.empty:
                print("No model pairs are significantly different at the adjusted level.")
            else:
                print("\nSignificant pairwise differences:")
                for _, row in sig_pairs.iterrows():
                    print(f"  {row['model_A']} vs {row['model_B']}: "
                          f"mean_diff = {row['mean_diff']:.4g}, "
                          f"p_adj = {row['p_value_adj']:.4g}")
            print("\nPairwise confidence intervals (bootstrap):")
            _ci_fig = _plot_ci_pairs(pairwise_df)
            if _ci_fig is not None:
                _safe_metric = re.sub(r"[^a-zA-Z0-9_]+", "_", metric_short)
                _ci_out = RESULTS_DIR / f"{PROJECT_NAME}_pairwise_ci_{_safe_metric}.png"
                _ci_fig.savefig(_ci_out, bbox_inches="tight", dpi=150)
                print(f"Saved: {_ci_out}")

    group_label_text = "all models" if group_label is None else group_label
    summary_text = _build_text_summary(
        summary_df=summary_df, pairwise_df=pairwise_df,
        metric_short=metric_short, data_label=data_label,
        group_label=group_label_text, alpha=alpha,
        higher_is_better=higher_is_better, friedman_p=p,
    )
    print("\nSummary:")
    print(summary_text)

def _on_run_button_clicked(_):
    """Handle 'Run statistical analysis' button click."""
    with stat_output:
        clear_output(wait=True)
        metric_short    = metric_dropdown.value
        data_source     = data_source_dropdown.value
        grouping_choice = stat_grouping_dropdown.value
        alpha           = float(alpha_float.value)

        if metric_short is None or metric_short == "<no metrics>":
            print("No metric selected. Please select a metric in Cell 11 first.")
            return

        df_runs = _build_run_level_df(metric_short=metric_short, source=data_source)
        if df_runs.empty:
            print(f"No run-level data found for metric '{metric_short}' "
                  f"and data source '{data_source}'.")
            return

        data_label = {"train": "training (validation)",
                      "infer": "inference (test)",
                      "both":  "training + inference"}[data_source]
        print(f"Running Friedman+Wilcoxon analysis for metric '{metric_short}' "
              f"using {data_label} runs.")
        print(f"Total runs in data set: {len(df_runs)}\n")

        if grouping_choice == "none":
            _analyze_one_subset(df=df_runs, metric_short=metric_short,
                                data_label=data_label, group_label=None, alpha=alpha)
        else:
            group_col = grouping_choice
            print(f"Comparing models within each group of '{group_col}'.\n")
            for group_value, sub_df in df_runs.groupby(group_col):
                if sub_df["model_base"].nunique() < 2:
                    print("=" * 80)
                    print(f"Group '{group_value}': fewer than 2 models present "
                          f"({sub_df['model_base'].nunique()} model). Skipping.")
                    continue
                _analyze_one_subset(df=sub_df, metric_short=metric_short,
                                    data_label=data_label, group_label=group_value, alpha=alpha)

run_button.on_click(_on_run_button_clicked)

In [ ]:
# === Cell 13: Visualization Widget ===

# --- Color palettes ---
CLASS_COLORS_RGB = {                  # prediction colors (solid, vivid)
    0: (255,   0,   0),               # body  - red
    1: (  0, 200,   0),               # cap   - green
    2: (  0,  80, 255),               # label - blue
}
GT_COLORS_RGB = {                     # ground-truth colors (lighter pastels)
    0: (255, 160, 160),               # body  - light pink/red
    1: (160, 255, 160),               # cap   - light green
    2: (160, 200, 255),               # label - light blue
}
DEFAULT_COLOR_RGB = (255, 255,   0)
DEFAULT_GT_RGB    = (255, 255, 160)
MASK_ALPHA        = 0.35
BOX_THICKNESS     = 2
TEXT_SCALE        = 0.55
TEXT_THICKNESS    = 2

CLASS_ID_TO_NAME = {v: k for k, v in CLASS_MAP.items()}

def _rgb2bgr(c):   return (c[2], c[1], c[0])
def _cls_bgr(cid): return _rgb2bgr(CLASS_COLORS_RGB.get(int(cid), DEFAULT_COLOR_RGB))
def _gt_bgr(cid):  return _rgb2bgr(GT_COLORS_RGB.get(int(cid), DEFAULT_GT_RGB))

# --- YOLO label parsing for GT overlay ---
def parse_yolo_label_line(line):
    """Parse one YOLO polygon label line into {'class_id', 'polygon_norm'}."""
    parts = line.strip().split()
    if len(parts) < 7:
        return None
    cls_id  = int(float(parts[0]))
    coords  = [float(v) for v in parts[1:]]
    polygon = list(zip(coords[0::2], coords[1::2]))
    return {"class_id": cls_id, "polygon_norm": polygon}

# --- Dashed polygon helper (OpenCV has no native dashed line) ---
def draw_dashed_polygon(img, points, color, thickness=2, dash_px=12, gap_px=7):
    """Draw a closed dashed polygon on img in-place."""
    pts = list(points)
    pts.append(pts[0])
    for i in range(len(pts) - 1):
        x0, y0 = int(pts[i][0]),   int(pts[i][1])
        x1, y1 = int(pts[i+1][0]), int(pts[i+1][1])
        dx, dy  = x1 - x0, y1 - y0
        seg_len = max(1, int(np.hypot(dx, dy)))
        step    = dash_px + gap_px
        t       = 0
        while t < seg_len:
            t_end = min(t + dash_px, seg_len)
            sx = int(round(x0 + dx * t     / seg_len))
            sy = int(round(y0 + dy * t     / seg_len))
            ex = int(round(x0 + dx * t_end / seg_len))
            ey = int(round(y0 + dy * t_end / seg_len))
            cv2.line(img, (sx, sy), (ex, ey), color, thickness, cv2.LINE_AA)
            t += step

# --- Single-image overlay renderer (predictions + GT dashed) ---
def draw_overlay(image_path, result,
                 show_masks=True, show_contours=True,
                 show_boxes=False, show_labels=True, show_gt=True):
    """Render predictions (solid) + GT (dashed lighter) on one image; return RGB."""
    img_bgr = result.orig_img.copy()
    overlay = img_bgr.copy()
    boxes   = result.boxes
    masks   = result.masks
    h, w    = img_bgr.shape[:2]

    # Filled prediction masks
    if show_masks and masks is not None and masks.xy is not None:
        for i, poly in enumerate(masks.xy):
            if poly is None or len(poly) < 3:
                continue
            cls_id = int(boxes.cls[i].item()) if boxes is not None and i < len(boxes) else -1
            pts    = np.array(poly, dtype=np.int32).reshape(-1, 1, 2)
            cv2.fillPoly(overlay, [pts], _cls_bgr(cls_id))
    blended = cv2.addWeighted(overlay, MASK_ALPHA, img_bgr, 1 - MASK_ALPHA, 0)

    # Solid prediction contours, boxes and labels
    if boxes is not None and len(boxes):
        xyxy    = boxes.xyxy.cpu().numpy()
        cls_ids = boxes.cls.cpu().numpy().astype(int)
        confs   = boxes.conf.cpu().numpy()
        polys   = masks.xy if masks is not None and masks.xy is not None else []

        for i in range(len(xyxy)):
            cls_id      = int(cls_ids[i])
            conf        = float(confs[i])
            color       = _cls_bgr(cls_id)
            x1, y1, x2, y2 = xyxy[i].astype(int)

            if show_contours and i < len(polys):
                poly = polys[i]
                if poly is not None and len(poly) >= 3:
                    pts = np.array(poly, dtype=np.int32).reshape(-1, 1, 2)
                    cv2.polylines(blended, [pts], True, color, 2, cv2.LINE_AA)
            if show_boxes:
                cv2.rectangle(blended, (x1, y1), (x2, y2), color, BOX_THICKNESS)
            if show_labels:
                name  = CLASS_ID_TO_NAME.get(cls_id, str(cls_id))
                label = f"{name} {conf:.2f}"
                (tw, th), bl = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX,
                                               TEXT_SCALE, TEXT_THICKNESS)
                ly1 = max(0, y1 - th - bl - 4)
                cv2.rectangle(blended, (x1, ly1), (x1 + tw + 6, y1), color, -1)
                cv2.putText(blended, label, (x1 + 3, y1 - 4),
                            cv2.FONT_HERSHEY_SIMPLEX, TEXT_SCALE,
                            (255, 255, 255), TEXT_THICKNESS, cv2.LINE_AA)

    # Dashed GT polygons from LABELS_DIR
    if show_gt:
        label_path = LABELS_DIR / (Path(image_path).stem + ".txt")
        if label_path.exists():
            with label_path.open("r", encoding="utf-8") as f:
                for line in f:
                    parsed = parse_yolo_label_line(line)
                    if parsed is None:
                        continue
                    pts_px = [(int(round(x * w)), int(round(y * h)))
                              for x, y in parsed["polygon_norm"]]
                    if len(pts_px) < 3:
                        continue
                    gt_color = _gt_bgr(parsed["class_id"])
                    draw_dashed_polygon(blended, pts_px, gt_color,
                                        thickness=2, dash_px=12, gap_px=7)

    return cv2.cvtColor(blended, cv2.COLOR_BGR2RGB)

# --- Best-run selection per model (used for multi-model comparison panels) ---
def safe_read_yaml(path):
    """Read a YAML file safely; returns None on failure."""
    try:
        with open(path, "r", encoding="utf-8") as f:
            return yaml.safe_load(f)
    except Exception:
        return None

def resolve_data_path(path_value, yaml_path):
    """Resolve a possibly-relative path from a dataset yaml."""
    if path_value is None:
        return None
    p = Path(path_value)
    if p.is_absolute():
        return p
    return (Path(yaml_path).parent / p).resolve()

def collect_candidate_images(trained_runs, allowed_splits=("train", "val")):
    """Walk TRAINED_RUNS data.yamls and collect all images from the requested splits."""
    image_exts   = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    found_images = []
    for run in trained_runs:
        data_yaml = run.get("data_yaml", None)
        if data_yaml is None:
            continue
        data_yaml = Path(data_yaml)
        if not data_yaml.exists():
            continue
        data_conf = safe_read_yaml(data_yaml)
        if not isinstance(data_conf, dict):
            continue
        for split in allowed_splits:
            split_path = resolve_data_path(data_conf.get(split, None), data_yaml)
            if split_path is None:
                continue
            if split_path.is_file():
                if split_path.suffix.lower() in image_exts:
                    found_images.append(split_path)
                else:
                    try:
                        with open(split_path, "r", encoding="utf-8") as f:
                            for line in f:
                                p = line.strip()
                                if not p:
                                    continue
                                candidate = Path(p)
                                if not candidate.is_absolute():
                                    candidate = (split_path.parent / candidate).resolve()
                                if candidate.exists() and candidate.suffix.lower() in image_exts:
                                    found_images.append(candidate)
                    except Exception:
                        pass
            elif split_path.is_dir():
                for p in split_path.rglob("*"):
                    if p.is_file() and p.suffix.lower() in image_exts:
                        found_images.append(p)
    return list(dict.fromkeys(found_images))

def get_best_metric_from_results_csv(run_dir, prefer_metric=BEST_MODEL_SELECTION_METRIC):
    """Return (metric_name, best_score) from a run's results.csv, or (None, None)."""
    results_csv = Path(run_dir) / "results.csv"
    if not results_csv.exists():
        return None, None
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    candidate_metrics = [prefer_metric, "metrics/mAP50-95(M)", "metrics/mAP50(M)",
                         "metrics/mAP50-95(B)", "metrics/mAP50(B)"]
    metric = next((m for m in candidate_metrics if m in df.columns), None)
    if metric is None:
        return None, None
    best_idx = pd.to_numeric(df[metric], errors="coerce").idxmax()
    return metric, float(df.loc[best_idx, metric])

def pick_best_run_per_model(trained_runs):
    """Return {model_base: best_run_metadata} keyed by BEST_MODEL_SELECTION_METRIC."""
    best_models = {}
    for run in trained_runs:
        weights_path = Path(run["run_dir"]) / "weights" / "best.pt"
        if not weights_path.exists():
            continue
        metric_name, metric_score = get_best_metric_from_results_csv(run["run_dir"])
        if metric_score is None:
            continue
        model_base = run["model_base"]
        if (model_base not in best_models) or (metric_score > best_models[model_base]["metric_score"]):
            best_models[model_base] = {
                "model_base":   model_base,
                "weights_path": weights_path,
                "run_dir":      Path(run["run_dir"]),
                "fold_name":    run["fold_name"],
                "metric_name":  metric_name,
                "metric_score": metric_score,
            }
    return best_models

# --- Multi-model rendering on one image (with GT overlay) ---
def render_model_comparison_for_image(image_path, best_models, conf, iou=0.50, imgsz=None,
                                       show_boxes=False, show_masks=True, show_labels=True,
                                       show_contours=True, show_gt=True,
                                       save_path=None, title_prefix="Image"):
    """Render overlays (predictions + dashed GT) from each best model side-by-side."""
    image_path     = Path(image_path)
    overlay_images = []
    titles         = []

    for model_base, info in best_models.items():
        model   = YOLO(str(info["weights_path"]))
        results = model.predict(
            source=str(image_path), conf=float(conf), iou=iou, imgsz=imgsz,
            retina_masks=True, verbose=False, save=False, device=DEVICE,
        )
        r       = results[0]
        rgb     = draw_overlay(image_path, r,
                               show_masks=show_masks, show_contours=show_contours,
                               show_boxes=show_boxes, show_labels=show_labels,
                               show_gt=show_gt)
        n_boxes = 0 if r.boxes is None else len(r.boxes)
        n_masks = 0 if r.masks is None else len(r.masks.data)
        overlay_images.append(rgb)
        titles.append(
            f"{model_base}\n"
            f"{info['fold_name']} | conf={float(conf):.2f}\n"
            f"boxes={n_boxes}, masks={n_masks}\n"
            f"{info['metric_name']}={info['metric_score']:.3f}"
        )

    if not overlay_images:
        print("No model overlays available.")
        return

    n     = len(overlay_images)
    ncols = min(3, n)
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, img, title in zip(axes, overlay_images, titles):
        ax.imshow(img); ax.set_title(title, fontsize=9); ax.axis("off")
    for ax in axes[len(overlay_images):]:
        ax.axis("off")
    plt.suptitle(
        f"{title_prefix}: {image_path.name}  |  "
        f"solid=prediction  dashed=GT  |  "
        f"body=red  cap=green  label=blue  (GT = lighter shade)",
        y=1.02, fontsize=12,
    )
    plt.tight_layout()
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, bbox_inches="tight", dpi=150)
        print(f"Saved: {save_path}")
    display(fig)
    plt.close(fig)

# --- Preconditions ---
if "TRAINED_RUNS" not in globals():
    raise RuntimeError("TRAINED_RUNS is not defined. Please run the training cell first.")

best_models = pick_best_run_per_model(TRAINED_RUNS)
if not best_models:
    raise RuntimeError("No best models found in TRAINED_RUNS (no valid weights/results.csv).")

CV_IMAGE_POOL   = [Path(p) for p in collect_candidate_images(TRAINED_RUNS, ("train", "val"))]
TEST_IMAGE_POOL = [Path(p) for p in read_non_empty_lines(TESTTXT)] if TESTTXT.exists() else []

TRAIN_RESULTS_DIR = RESULTS_DIR
INFER_RESULTS_DIR = INFERENCE_DIR / "results"
INFER_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if not CV_IMAGE_POOL and not TEST_IMAGE_POOL:
    print("No images available for visualization (neither CV nor Test).")
else:
    mode_toggle = widgets.ToggleButtons(
        options=[("CV Images", "cv"), ("Test Images", "test")],
        value="cv" if CV_IMAGE_POOL else "test",
        description="Mode:", button_style="",
    )
    def _make_opts(pool):
        return [(f"{i}: {Path(p).name}", i) for i, p in enumerate(pool)]

    _init_pool     = CV_IMAGE_POOL if mode_toggle.value == "cv" else TEST_IMAGE_POOL
    image_dropdown = widgets.Dropdown(
        options=_make_opts(_init_pool), description="Image:",
        layout=widgets.Layout(width="700px"),
    )

    show_button   = widgets.Button(description="Show",     button_style="primary", icon="eye")
    random_button = widgets.Button(description="Random",   button_style="info",    icon="random")
    save_button_v = widgets.Button(description="Save PNG", icon="save")

    conf_slider = widgets.FloatSlider(
        value=CONF_THRESHOLD, min=0.0, max=1.0, step=0.01,
        description="Conf:", readout_format=".2f",
        layout=widgets.Layout(width="350px"),
    )
    iou_slider = widgets.FloatSlider(
        value=IOU_THRESHOLD, min=0.10, max=0.90, step=0.05,
        description="IoU:", readout_format=".2f",
        layout=widgets.Layout(width="350px"),
    )

    show_masks_cb    = widgets.Checkbox(value=True,  description="Masks")
    show_contours_cb = widgets.Checkbox(value=True,  description="Contours")
    show_boxes_cb    = widgets.Checkbox(value=False, description="Boxes")
    show_labels_cb   = widgets.Checkbox(value=True,  description="Labels")
    show_gt_cb       = widgets.Checkbox(value=True,  description="Ground truth (dashed)")

    viewer_output = widgets.Output()

    def _current_pool():   return CV_IMAGE_POOL if mode_toggle.value == "cv" else TEST_IMAGE_POOL
    def _current_prefix(): return "CV image"    if mode_toggle.value == "cv" else "Test image"

    def _render(save_path=None):
        pool = _current_pool()
        idx  = image_dropdown.value
        if idx is None or idx < 0 or idx >= len(pool):
            return
        image_path = pool[idx]
        with viewer_output:
            clear_output(wait=True)
            render_model_comparison_for_image(
                image_path=image_path, best_models=best_models,
                conf=conf_slider.value, iou=iou_slider.value, imgsz=IMAGE_SIZE,
                show_boxes=show_boxes_cb.value, show_masks=show_masks_cb.value,
                show_labels=show_labels_cb.value, show_contours=show_contours_cb.value,
                show_gt=show_gt_cb.value,
                save_path=save_path,
                title_prefix=_current_prefix(),
            )

    def on_mode_changed(change):
        """Repopulate the image dropdown when the CV/Test toggle changes."""
        if change["name"] != "value":
            return
        pool = _current_pool()
        if not pool:
            with viewer_output:
                clear_output(wait=True)
                print(f"No images available for mode '{change['new']}'.")
            image_dropdown.options = []
            image_dropdown.value   = None
        else:
            image_dropdown.options = _make_opts(pool)
            image_dropdown.value   = 0

    mode_toggle.observe(on_mode_changed)
    show_button.on_click(lambda _: _render())
    random_button.on_click(lambda _: (
        setattr(image_dropdown, "value", random.randrange(len(_current_pool()))),
        _render(),
    ))
    def _on_save_clicked(_):
        pool = _current_pool()
        if not pool:
            return
        idx = image_dropdown.value
        if idx is None or idx < 0 or idx >= len(pool):
            return
        image_path = pool[idx]
        if mode_toggle.value == "cv":
            out_dir = TRAIN_RESULTS_DIR; tag = "cvviz"
        else:
            out_dir = INFER_RESULTS_DIR; tag = "testviz"
        out_dir.mkdir(parents=True, exist_ok=True)
        save_path = out_dir / f"{tag}_{Path(image_path).stem}.png"
        _render(save_path=save_path)

    save_button_v.on_click(_on_save_clicked)

    display(
        widgets.VBox([
            widgets.HBox([mode_toggle]),
            widgets.HBox([image_dropdown]),
            widgets.HBox([show_button, random_button, save_button_v]),
            widgets.HBox([conf_slider, iou_slider]),
            widgets.HBox([show_masks_cb, show_contours_cb, show_boxes_cb,
                          show_labels_cb, show_gt_cb]),
        ]),
        viewer_output,
    )
    on_mode_changed({"name": "value", "new": mode_toggle.value})